Propósito geral do notebook/script
----------------------------------
Este notebook realiza duas análises principais em um projeto de RNA‑seq focado em TNBC:
1) **Processamento: lncRNA** – carrega a matriz de contagens por gene, integra a anotação
do GTF (com lncRNAs), computa métricas (CPM, genes detectados), e gera figuras
descritivas (distribuições por biotipo, frações por amostra, top lncRNAs).
2) **Processamento: WGCNA (atualizado)** – carrega os arquivos exportados do WGCNA no
formato Cytoscape (nodes/edges), estima automaticamente um *cutoff* de peso para manter
~5% das arestas (percentil 95), filtra a rede, produz estatísticas estruturais,
contabiliza interações por biotipo, e anota *hub genes* com termos GO via MyGene.info.

### Comum

In [ ]:
!pip install gseapy

In [ ]:
!pip install biopython

In [ ]:
# Usado na normalização
try:
    import rpy2.robjects as ro
    from rpy2.robjects import pandas2ri
    pandas2ri.activate()
    have_rpy2 = True
except Exception:
    have_rpy2 = False

if not have_rpy2:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "rpy2>=3.5.1"])
    import rpy2.robjects as ro
    from rpy2.robjects import pandas2ri
    pandas2ri.activate()

# Instala edgeR se necessário
ro.r('if (!requireNamespace("BiocManager", quietly=TRUE)) install.packages("BiocManager", repos="https://cloud.r-project.org")')
ro.r('if (!requireNamespace("edgeR", quietly=TRUE)) BiocManager::install("edgeR", update=FALSE, ask=FALSE)')

In [ ]:
# ============================
# IMPORTS
# ============================import pandas as pd
from google.colab import drive
import os
import re
import json
import pandas as pd
import numpy as np
import seaborn as sns
import gc
import math
import itertools
from scipy import stats
from statsmodels.stats.multitest import multipletests
from sklearn.decomposition import PCA
from collections import Counter, defaultdict
import requests
from matplotlib_venn import venn2, venn2_unweighted
from matplotlib import pyplot as plt
from pathlib import Path
import gzip
from Bio import SeqIO
from Bio.Seq import Seq


In [ ]:
# ============================
# FUNÇÕES UTILITÁRIAS
# ============================
def parse_gtf_attributes(attribute_string, keys_to_extract):
  """Extrai pares chave->valor do campo 'attribute' de um GTF.

  Parâmetros
  ----------
  attribute_string : str
  Campo de atributos do GTF (ex.: 'gene_id "ENSG..."; gene_name "TP53"; ...').
  keys_to_extract : list[str]
  Lista de chaves desejadas (ex.: ['gene_id','gene_name','gene_type']).

  Retorna
  -------
  dict
  Dicionário contendo apenas as chaves pedidas, quando presentes.
  """
  pattern = re.compile(r'(\w+)\s+"([^"]+)"')
  extracted = {}
  for k, v in pattern.findall(str(attribute_string)):
      if k in keys_to_extract:
          extracted[k] = v
  return extracted

def tidy_counts_by_biotype(df, sample_cols, biotype_col='gene_type'):
  """Agrega contagens por amostra × biotipo e calcula fração por amostra.

  Parâmetros
  ----------
  df : pd.DataFrame
  DataFrame com colunas de contagens por amostra e uma coluna de biotipo.
  sample_cols : list[str]
  Lista com os nomes das colunas de amostra.
  biotype_col : str, padrão 'gene_type'
  Nome da coluna com o biotipo do gene (ex.: 'lncRNA', 'protein_coding').

  Retorna
  -------
  pd.DataFrame
  Tabela no formato *long* com colunas: 'sample', biotype_col, 'counts',
  'total_counts' e 'fraction' (fração dentro da amostra).
  """
  melted = df.melt(id_vars=[biotype_col], value_vars=sample_cols,
                    var_name='sample', value_name='counts')
  by = melted.groupby(['sample', biotype_col], as_index=False)['counts'].sum()
  totals = by.groupby('sample', as_index=False)['counts'].sum().rename(columns={'counts':'total_counts'})
  out = by.merge(totals, on='sample', how='left')
  out['fraction'] = np.where(out['total_counts']>0, out['counts']/out['total_counts'], 0.0)
  return out

def make_cpm(counts_df, sample_cols):
  """Calcula CPM (counts per million) por coluna de amostra.

  Parâmetros
  ----------
  counts_df : pd.DataFrame
  Matriz de contagens com uma linha por gene e colunas de amostras.
  sample_cols : list[str]
  Nomes das colunas de amostra.

  Retorna
  -------
  (pd.DataFrame, pd.Series)
  - DataFrame de CPM com mesma forma que `counts_df[sample_cols]`.
  - Série com *library sizes* por amostra, após substituir zeros por NaN.
  """
  lib_sizes = counts_df[sample_cols].sum(axis=0)
  lib_sizes = lib_sizes.replace(0, np.nan)
  cpm = counts_df[sample_cols].div(lib_sizes, axis=1) * 1e6
  return cpm, lib_sizes

def load_gtf_gene_biotype(gtf_file):
  """Lê um GTF e retorna um mapa gene_name → gene_type, priorizando biotipos.


  Estratégia:
  - Lê o GTF e *normaliza* o campo attribute.
  - Aceita tanto `gene_type` quanto `gene_biotype`.
  - Filtra apenas features `gene` e remove duplicatas.
  - Quando um gene_name tem múltiplos biotipos (raro), prioriza: lncRNA > protein_coding > outros.


  Retorna DataFrame com colunas ['gene_name','gene_type'].
  """
  cols = ['seqname','source','feature','start','end','score','strand','frame','attribute']
  g = pd.read_csv(gtf_file, sep='\t', comment='#', header=None, names=cols)

  # extrair atributos do GTF (robusto a gene_type vs gene_biotype)
  desired = ['gene_id','gene_name','gene_type','gene_biotype']
  attr = pd.json_normalize(g['attribute'].apply(lambda s: parse_gtf_attributes(s, desired)))

  # padronizar para a coluna 'gene_type'
  if 'gene_type' not in attr or attr['gene_type'].isna().all():
      attr['gene_type'] = attr.get('gene_biotype')

  g = pd.concat([g.drop(columns=['attribute']), attr], axis=1)

  # manter apenas nível gene
  g = (g[g['feature']=='gene'][['gene_name','gene_type']]
          .dropna()
          .drop_duplicates())

  # resolver nomes com biotipos múltiplos (prioriza lncRNA > protein_coding > outros)
  rank = {'lncRNA': 2, 'protein_coding': 1}
  g['rank'] = g['gene_type'].map(rank).fillna(0)
  g = g.sort_values(['gene_name','rank'], ascending=[True,False]).drop_duplicates('gene_name')

  return g[['gene_name','gene_type']]

def parse_traits_from_sample(sample):
  """Extrai traços/condições do nome de amostra (útil para análise posterior).


  Retorna um dicionário binário com chaves: DEX, BRM014, SIGATA6, VEH, DMSO, H4, H24.
  - Usa *string matching* simples; garanta convenção consistente nos nomes de amostras.
  """
  s = sample.upper()
  traits = dict(DEX=0, BRM014=0, SIGATA6=0, VEH=0, DMSO=0, H4=0, H24=0)
  if 'DEX' in s: traits['DEX']=1
  if 'BRM014' in s: traits['BRM014']=1
  if 'SIGATA6' in s: traits['SIGATA6']=1
  if 'VEH' in s: traits['VEH']=1
  if 'DMSO' in s: traits['DMSO']=1
  if '4H' in s: traits['H4']=1
  if '24H' in s: traits['H24']=1
  return traits

def summarize_detection(mask, label):
  """Resume a detecção de genes para um critério dado por `mask`.

  Parâmetros
  ----------
  mask : pd.Series[bool]
  *Boolean mask* indicando quais linhas/genes são considerados detectados.
  label : str
  Rótulo descritivo do critério (usado em gráficos e tabelas).

  Retorna
  -------
  dict
  Contém: total de genes detectados, e subtotais por biotipo (lncRNA, protein_coding, outros).
  """
  sub = merged[mask]
  out = {
      'label': label,
      'n_genes_detected': int(sub.shape[0]),
      'n_lnc_detected':   int(sub[sub['gene_type']=='lncRNA'].shape[0]),
      'n_pc_detected':    int(sub[sub['gene_type']=='protein_coding'].shape[0]),
      'n_other_detected': int(sub[~sub['gene_type'].isin(['lncRNA','protein_coding'])].shape[0]),
  }
  return out

def fetch_go_terms_mygene(symbol_or_ensg):
  """Consulta termos GO (BP/MF/CC) via MyGene.info para um gene humano.

  Parâmetros
  ----------
  symbol_or_ensg : str
  Símbolo do gene (ex.: 'TP53') ou ENSG (ex.: 'ENSG00000141510').

  Retorna
  -------
  tuple(list, list, list)
  Três listas de termos para GO: BP, MF e CC.

  Comentários
  --------
  - Em caso de múltiplos hits, pega o primeiro.
  """
  base = "https://mygene.info/v3/query"
  q = f"symbol:{symbol_or_ensg}"
  if str(symbol_or_ensg).upper().startswith('ENSG'):
      q = f"ensembl.gene:{symbol_or_ensg}"
  params = {"q": q, "species": "human", "fields": "symbol,name,go.BP.term,go.MF.term,go.CC.term", "size": 1}
  try:
      r = requests.get(base, params=params, timeout=20)
      r.raise_for_status()
      hits = r.json().get('hits', [])
      if not hits:
          return [],[],[]
      hit = hits[0]
      def _collect(branch):
          terms = hit.get('go', {}).get(branch, [])
          if isinstance(terms, dict): terms = [terms]
          return [t.get('term','') for t in terms if isinstance(t, dict) and 'term' in t]
      return _collect('BP'), _collect('MF'), _collect('CC')
  except Exception:
      return [],[],[]

def symbols_to_nodes(symbols, nodes_df):
    """Mapeia símbolos (UPPER) -> nodeName existentes na tabela base_nodes."""
    syms = {s.upper() for s in symbols}
    return set(nodes_df.loc[nodes_df['gene_name'].astype(str).str.upper().isin(syms), 'nodeName'])

def load_edges(path):
    return pd.read_csv(path, sep='\t', dtype={'fromNode':str, 'toNode':str})

def export_cytoscape(edges_df, nodes_df, out_prefix, extra_nodes=None):
    """
    Exporta edges e nodes para Cytoscape.
    - extra_nodes: nós adicionais a incluir mesmo sem arestas (ex.: GATA6)
    - Não duplica nós; usa união de conjuntos
    """
    node_set = set(edges_df['fromNode']) | set(edges_df['toNode'])
    if extra_nodes:
        node_set |= set(extra_nodes)

    sub_nodes = nodes_df[nodes_df['nodeName'].isin(node_set)].copy()

    # nodes
    sub_nodes[['nodeName','gene_name','module','biotype']].to_csv(
        f"{out_prefix}_nodes.tsv", sep="\t", index=False
    )
    # edges (apenas colunas necessárias)
    edges_df[['fromNode','toNode','weight']].to_csv(
        f"{out_prefix}_edges.tsv", sep="\t", index=False
    )


def network_metrics(edges_df, nodes_df):
    if edges_df.empty:
        return dict(N=0,E=0,avg_degree=np.nan,density=np.nan,
                    pct_lnc=np.nan,has_GATA6=False,has_GATA6_AS1=False)
    keep_nodes = set(edges_df['fromNode']) | set(edges_df['toNode'])
    nn = nodes_df[nodes_df['nodeName'].isin(keep_nodes)].copy()
    N = nn.shape[0]; E = edges_df.shape[0]
    avg_degree = (2*E)/N if N>0 else np.nan
    density = (2*E)/(N*(N-1)) if N>1 else np.nan
    pct_lnc = (nn['biotype'].eq('lncRNA').sum()/N*100) if N>0 else np.nan
    has_g6  = (nn['gene_name'].str.upper()=='GATA6').any()
    has_g6a = (nn['gene_name'].str.upper()=='GATA6-AS1').any()
    return dict(N=N,E=E,avg_degree=avg_degree,density=density,
                pct_lnc=pct_lnc,has_GATA6=bool(has_g6),has_GATA6_AS1=bool(has_g6a))

def save_metrics(label, edges_df, nodes_df, extra_nodes=None):
    """
    Métricas de estrutura calculadas sobre o conjunto de nós
    = nós das arestas U extra_nodes (se fornecido).
    """
    node_set = set(edges_df['fromNode']) | set(edges_df['toNode'])
    if extra_nodes:
        node_set |= set(extra_nodes)
    sub_nodes = nodes_df[nodes_df['nodeName'].isin(node_set)].copy()

    N = sub_nodes.shape[0]
    E = edges_df.shape[0]
    avg_degree = (2*E/N) if N > 0 else 0.0
    density = (2*E)/(N*(N-1)) if N > 1 else 0.0

    # biotipos
    n_lnc = int((sub_nodes['biotype'] == 'lncRNA').sum())
    n_pc  = int((sub_nodes['biotype'] == 'protein_coding').sum())
    pct_lnc = 100.0 * (n_lnc / N) if N > 0 else 0.0

    # presença de GATA6 / GATA6-AS1 (independente de haver edges)
    genes_up = sub_nodes['gene_name'].astype(str).str.upper()
    has_gata6    = any(genes_up == 'GATA6')
    has_gata6_as = any(genes_up.isin(['GATA6-AS1','AS1-GATA6']))

    return pd.DataFrame([{
        'filter': label,
        'N': N, 'E': E,
        'avg_degree': avg_degree,
        'density': density,
        'pct_lnc': pct_lnc,
        'has_GATA6': has_gata6,
        'has_GATA6_AS1': has_gata6_as
    }])

def topk_per_node(edges_df, k=5):
    e1 = edges_df.sort_values('weight', ascending=False).groupby('fromNode', as_index=False).head(k)
    e2 = edges_df.sort_values('weight', ascending=False).groupby('toNode',   as_index=False).head(k)
    return pd.concat([e1,e2], ignore_index=True).drop_duplicates()

In [ ]:
# ============================
# MONTAGEM DO GOOGLE DRIVE (Colab)
# ============================
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# drive.mount("/content/drive", force_remount=True)

In [ ]:
# ============================
# PARÂMETROS E CAMINHOS DO PROJETO
# ============================
SOURCE_FOLDER = '/content/drive/MyDrive/Saude_lncRNA'
PROCESSED_DIR = f'{SOURCE_FOLDER}/data/processed'
NOTEBOOK_PROCESSED_DIR = f'{PROCESSED_DIR}/notebook'
KNOWN_DIR = f"{SOURCE_FOLDER}/data/processed/known_sets"
DIFFEXP_DIR = f"{SOURCE_FOLDER}/data/processed/diff_expression/tables/differential"

# Arquivos de entrada
COUNTS_TSV = f'{PROCESSED_DIR}/salmon.merged.gene_counts.tsv'
CHIPSEQ_COUNTS_TSV   = f'{PROCESSED_DIR}/chip_seq_salmon.merged.transcript_counts.tsv'
GTF_FILE     = f'{PROCESSED_DIR}/Anotacao_com_lncRNA.gtf'
NODE_FILE  = f'{PROCESSED_DIR}/CytoscapeInput-nodes.txt'
EDGE_FILE  = f'{PROCESSED_DIR}/CytoscapeInput-edges.txt'
NORMALIZED_NODE_FILE = f"{PROCESSED_DIR}/CytoscapeInput-nodes_normalized.txt"
NORMALIZED_EDGE_FILE = f"{PROCESSED_DIR}/CytoscapeInput-edges_normalized.txt"
HUBS_CSV = f"{PROCESSED_DIR}/hubs.csv"
GENOME_FA = f"{SOURCE_FOLDER}/data/external and raw/GRCh38.primary_assembly.genome.fa.gz"

# Arquivos de entrada/saída para o filtro F (Terceira entrega) - Literatura
PEAKS_GSE47535 = f"{KNOWN_DIR}/GSE47535/GSM1151694_Gata6-2a_norm_peaks_FDR1.txt"
OUT_LIT_LIST   = f"{KNOWN_DIR}/literature_gata6_network_genes.txt"
OUT_GSE_LIST   = f"{KNOWN_DIR}/GSE47535/genes_from_GSM1151694_TSS10kb.txt"  # lista extra para auditoria

# Criação de diretórios
FIG_DIR      = f'{SOURCE_FOLDER}/figures' # Diretório principal as figuras serão salvas

LNCRNA_FIG_DIR         = f'{FIG_DIR}/lncrna' #Sub diretório para figuras lncRNA
os.makedirs(LNCRNA_FIG_DIR, exist_ok=True)

CHIPSEQ_LNCRNA_FIG_DIR         = f'{FIG_DIR}/lncrna/chip-seq' #Sub diretório para figuras lncRNA-ChipSeq
os.makedirs(CHIPSEQ_LNCRNA_FIG_DIR, exist_ok=True)

WGCNA_FIG_DIR         = f'{FIG_DIR}/wgcna' #Sub diretório para figuras WGCNA
os.makedirs(WGCNA_FIG_DIR, exist_ok=True)

VENN_FIG_DIR = f"{FIG_DIR}/venn"
os.makedirs(VENN_FIG_DIR, exist_ok=True)

SUMMARY_EXPORT_DIR = f"{NOTEBOOK_PROCESSED_DIR}/slide_helpers_and_summaries"
os.makedirs(SUMMARY_EXPORT_DIR, exist_ok=True)

MODULE_EXPORT_DIR = f"{NOTEBOOK_PROCESSED_DIR}/wgcna/modules" #Diretório para armazenar os edge/node .tsv de cada módulo
os.makedirs(MODULE_EXPORT_DIR, exist_ok=True)

WGCNA_FILTERED_EXPORT_DIR = f"{NOTEBOOK_PROCESSED_DIR}/wgcna/filtered" #Diretório para armazenar os edge/node .tsv filtrados
os.makedirs(WGCNA_FILTERED_EXPORT_DIR, exist_ok=True)

FILTER_EXPORT_DIR = f"{NOTEBOOK_PROCESSED_DIR}/filters"
os.makedirs(FILTER_EXPORT_DIR, exist_ok=True)

FILTER_DEG_EXPORT_DIR = f"{NOTEBOOK_PROCESSED_DIR}/filters/d_deg"
os.makedirs(FILTER_DEG_EXPORT_DIR, exist_ok=True)

FILTER_ENRICH_EXPORT_DIR = f"{NOTEBOOK_PROCESSED_DIR}/filters/enrichment"
os.makedirs(FILTER_ENRICH_EXPORT_DIR, exist_ok=True)

INTA_EXPORT_DIR = f"{NOTEBOOK_PROCESSED_DIR}/filters/inta_inputs"
os.makedirs(INTA_EXPORT_DIR, exist_ok=True)

DEG_SUMMARIES_EXPORT_DIR = f"{FILTER_DEG_EXPORT_DIR}/deg_summaries_padj_only"
os.makedirs(DEG_SUMMARIES_EXPORT_DIR, exist_ok=True)

# Diretórios de entrada/saída específicos do WGCNA
WGCNA_INDIR   = PROCESSED_DIR
WGCNA_OUTDIR  = WGCNA_FIG_DIR

# Hiperparâmetros e thresholds usados em múltiplas etapas
MIN_COUNT = 10 # número mínimo de reads em uma amostra para contar como detectado
MIN_SAMPLES = 1 # número mínimo de amostras atingindo MIN_COUNT

PERCENTILE_THRESHOLD = 0.95 # 95th percentile
PERCENTILE_THRESHOLD_B_FILTER = 0.99 #99th percentile

TARGET_GENES = {'GATA6','GATA6-AS1'} # Genes para usar nos hubs/modulos

EXTRA_SYMBOLS_ALWAYS = ['GATA6', 'GATA6-AS1', 'AS1-GATA6']  # Usado para forçar presença de GATA6 / GATA6-AS1 em todos os filtros (Terceira entrega)


D_FILTER_PADJ_FILTER = 0.05 # Filtro Padj usado no filtro D (Terceira entrega)
D_FILTER_ABSLFC_FILTER = 0.0 # Filtro |Log2FC| usado no filtro D (Terceira entrega)


### Processamento: lncRNA

In [ ]:
# ==========================================
# BLOCO: Processamento — lncRNA (descrição)
# ==========================================
# Este bloco integra contagens com a anotação, computa CPM, sumariza detecção
# e gera figuras descritivas (A–D) em FIG_DIR/lncrna.

#### lncRNA

In [ ]:
# ---------- 1) Ler counts ----------
gene_df = pd.read_csv(COUNTS_TSV, sep='\t')

# Identificar colunas de amostra
fixed_cols = {'gene_id', 'gene_name'}
sample_cols = [c for c in gene_df.columns if c not in fixed_cols]

# Remover versões do gene_id (ex.: ENSG000001.1 -> ENSG000001)
gene_df['gene_id_base'] = gene_df['gene_id'].astype(str).str.split('.').str[0]

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Saude_lncRNA/data/processed/salmon.merged.gene_counts.tsv'

In [ ]:
# ---------- 2) Ler GTF (somente 'gene') e extrair atributos ----------
# Definir nomes de colunas padrão para GTF e ler o arquivo ignorando comentários (#)
gtf_cols = ['seqname','source','feature','start','end','score','strand','frame','attribute']

gtf_raw = pd.read_csv(GTF_FILE, sep='\t', comment='#', header=None, names=gtf_cols)

# Extrair apenas as chaves desejadas do campo attribute
desired_keys = ['gene_id','gene_name','gene_type']
attr = gtf_raw['attribute'].apply(lambda s: parse_gtf_attributes(s, desired_keys))
attr_df = pd.json_normalize(attr)

gtf_df = pd.concat([gtf_raw.drop(columns=['attribute']), attr_df], axis=1)

# Filtrar somente a feature=gene, remove duplicatas e pega colunas essenciais
gtf_genes = (gtf_df[gtf_df['feature']=='gene']
             [['gene_id','gene_name','gene_type']]
             .drop_duplicates())

# Remove versões do gene_id para compatibilizar com a contagem
gtf_genes['gene_id_base'] = gtf_genes['gene_id'].astype(str).str.split('.').str[0]

In [ ]:
# ---------- 3) Merge counts + anotação ----------
# Join por gene_id_base e traz gene_type/gene_name do GTF
merged = (gene_df
          .merge(gtf_genes[['gene_id_base','gene_type','gene_name']],
                 on='gene_id_base', how='left', suffixes=('', '_gtf')))

# Priorizar gene_name do counts quando existir; caso contrário, usar do GTF
merged['gene_name_final'] = merged['gene_name'].fillna(merged['gene_name_gtf'])
merged['gene_type'] = merged['gene_type'].fillna('unknown')

In [ ]:
# ---------- 4) Normalização TMM (edgeR) → logCPM ----------
# Prepara matriz de contagens (genes x amostras) para o R
counts_mat = merged[sample_cols].copy()
counts_mat.index = merged['gene_name_final'].fillna(merged['gene_id']).astype(str)

ro.globalenv['counts_df'] = pandas2ri.py2rpy(counts_mat)
ro.r("""
suppressPackageStartupMessages(library(edgeR))
dge <- DGEList(counts = as.matrix(counts_df))
# TMM
dge <- calcNormFactors(dge, method = "TMM")
# logCPM com prior.count=1 (estável para baixa contagem)
logCPM <- cpm(dge, log = TRUE, prior.count = 1)
""")

logCPM_df = pandas2ri.rpy2py(ro.r('as.data.frame(logCPM)'))
logCPM_df.index.name = 'gene'
logCPM_df.columns = sample_cols

In [ ]:
# Anexa média de logCPM ao merged (para gráficos/ordenações)
merged['mean_logCPM'] = logCPM_df.reindex(counts_mat.index).mean(axis=1).values

In [ ]:
# ---------- 5) Definir "detectados" ----------
# Critério 1 (liberal): pelo menos 1 read em qualquer amostra
merged['total_counts'] = merged[sample_cols].sum(axis=1)
detected_any = merged['total_counts'] > 0

# Critério 2 (stringente): >= MIN_COUNT reads em pelo menos MIN_SAMPLES amostras
detected_stringent = (merged[sample_cols] >= MIN_COUNT).sum(axis=1) >= MIN_SAMPLES

In [ ]:
# ---------- 6) Métricas globais ----------
# Resumos por critério — número de genes detectados por biotipo
summary_any = summarize_detection(detected_any, 'any_read>0')
summary_str = summarize_detection(detected_stringent, f'>={MIN_COUNT} reads in >={MIN_SAMPLES} samples')


# Frações de reads por biotipo (global, somando amostras)
reads_by_biotype = merged.groupby('gene_type')[sample_cols].sum()
global_reads = reads_by_biotype.sum(axis=1).sort_values(ascending=False)
global_total = global_reads.sum()

# Fração total por biotipo (útil para checar o peso de lncRNA vs PC)
global_fracs = (global_reads / global_total).rename('fraction')

In [ ]:
# ---------- 7) CPM clássico (não-normalizado) para compatibilidade com v0 dos gráficos ----------
# CPM médio por gene (média nas colunas de amostra)
cpm_raw, lib_sizes = make_cpm(merged, sample_cols)
merged['mean_cpm'] = cpm_raw.mean(axis=1)

In [ ]:
# ---------- 8) Tabelas auxiliares para gráficos ----------

# (a) Tabela longa para barplot: genes detectados por biotipo × critério
det_tab = pd.DataFrame([summary_any, summary_str])
det_long = det_tab.melt(id_vars=['label'],
                        value_vars=['n_lnc_detected','n_pc_detected','n_other_detected'],
                        var_name='class', value_name='n')
det_long['class'] = det_long['class'].map({'n_lnc_detected':'lncRNA',
                                           'n_pc_detected':'protein_coding',
                                           'n_other_detected':'outros'})
det_long['label'] = det_long['label'].map({'any_read>0':'Qualquer leitura (>0)',
                                          f'>={MIN_COUNT} reads in >={MIN_SAMPLES} samples':
                                          f'Robusto (≥{MIN_COUNT} reads em ≥{MIN_SAMPLES} amostras)'})


# (b) Fração por amostra (stacked 100%)
# Constrói tabela por amostra×biotipo e colapsa biotipos menos comuns em "outros"
tidy = tidy_counts_by_biotype(merged, sample_cols, 'gene_type')

main_biotypes = ['protein_coding','lncRNA']
tidy_plot = tidy.copy()
tidy_plot['biotype'] = np.where(tidy_plot['gene_type'].isin(main_biotypes),
                                tidy_plot['gene_type'], 'outros')
tidy_plot = (tidy_plot.groupby(['sample','biotype'], as_index=False)
             [['counts','total_counts']].sum())
tidy_plot['fraction'] = np.where(tidy_plot['total_counts']>0,
                                 tidy_plot['counts']/tidy_plot['total_counts'], 0.0)

# (c) Distribuição de expressão (log10 CPM) para lncRNA vs protein_coding
expr_dist = (merged[merged['gene_type'].isin(['lncRNA','protein_coding'])]
             [['gene_type','mean_logCPM']].copy())

expr_dist['log10_mean_cpm_norm'] = expr_dist['mean_logCPM']  # já é logCPM


In [ ]:
# ---------- 8) Gráficos ----------
# Configuração estética global do seaborn/matplotlib
sns.set_theme(style='whitegrid', context='talk')

# A) Genes detectados por biotipo (dois critérios)
plt.figure(figsize=(10,6))
ax = sns.barplot(data=det_long, x='label', y='n', hue='class',
                 palette={'protein_coding':'#4c78a8','lncRNA':'#f58518','outros':'#54a24b'})
ax.set_xlabel('')
ax.set_ylabel('Genes detectados')
ax.set_title('Genes detectados por biotipo')
ax.legend(title='Biotipo', frameon=False)

plt.tight_layout()

plt.savefig(f'{LNCRNA_FIG_DIR}/A_genes_detectados_biotipo.png', dpi=300)
plt.close()

In [ ]:
# B) Fração de reads por amostra (stacked 100%)
# montar barras empilhadas 100%
samples = tidy_plot['sample'].unique()
biotypes = ['protein_coding','lncRNA','outros']
plot_mat = (tidy_plot.pivot(index='sample', columns='biotype', values='fraction')
            .reindex(columns=biotypes).fillna(0.0))
plot_mat = plot_mat.sort_index()

colors = {'protein_coding':'#4c78a8','lncRNA':'#f58518','outros':'#54a24b'}
plt.figure(figsize=(max(12, 0.5*len(samples)), 6))
bottom = np.zeros(len(plot_mat))
for bt in biotypes:
    vals = plot_mat[bt].values
    plt.bar(plot_mat.index, vals, bottom=bottom, color=colors[bt], label=bt)
    bottom += vals
plt.xticks(rotation=75, ha='right')
plt.ylim(0,1)
plt.ylabel('Fração de reads')
plt.title('Fração de reads por biotipo em cada amostra')
plt.legend(title='Biotipo', frameon=False)
plt.tight_layout()

plt.savefig(f'{LNCRNA_FIG_DIR}/B_fracao_reads_por_amostra.png', dpi=300)
plt.close()

In [ ]:
# C) Distribuição de expressão (mean CPM) lncRNA vs protein_coding
plt.figure(figsize=(8,6))

ax = sns.violinplot(data=expr_dist, x='gene_type', y='log10_mean_cpm_norm',
                    palette={'protein_coding':'#4c78a8','lncRNA':'#f58518'}, cut=0)

# Overlay com pontos (amostra de até 2000 genes para não sobrecarregar o gráfico)
sns.stripplot(data=expr_dist.sample(min(2000, expr_dist.shape[0]), random_state=42),
              x='gene_type', y='log10_mean_cpm_norm', color='k', size=1, alpha=0.3, jitter=0.2)
ax.set_xlabel('')
ax.set_ylabel('mean logCPM (TMM)')
ax.set_title('Distribuição de expressão (TMM → logCPM)')
plt.tight_layout()

plt.savefig(f'{LNCRNA_FIG_DIR}/C_distribuicao_expressao_log10meanCPM.png', dpi=300)
plt.close()

In [ ]:
# D) Top 20 lncRNAs (por expressão média CPM)
plt.figure(figsize=(8,10))

top_lnc = (merged[merged['gene_type']=='lncRNA']
           .sort_values('mean_logCPM', ascending=False)
           .loc[:, ['gene_id','gene_name_final','mean_logCPM']].head(20))

top_lnc_plot = top_lnc.iloc[::-1]  # inverter para barras horizontais de baixo para cima
labels = top_lnc_plot['gene_name_final'].fillna(top_lnc_plot['gene_id'])
plt.barh(labels, top_lnc_plot['mean_logCPM'], color='#f58518')
plt.xlabel('mean logCPM (TMM)')
plt.title('Top 20 lncRNAs por expressão (TMM → logCPM)')
plt.tight_layout()

plt.savefig(f'{LNCRNA_FIG_DIR}/D_top20_lncRNAs_meanCPM.png', dpi=300)
plt.close()

In [ ]:
# ---------- 9) Conjunto de lncRNAs "robustos" para usar no WGCNA ----------
robust_lnc_set = set(merged.loc[(merged['gene_type']=='lncRNA') & detected_stringent, 'gene_name_final'].str.upper())

In [ ]:
# === Slide helper: numbers for "Processamento: lncRNA" ===
# Uses variables already created above:
# - gene_df, merged, sample_cols
# - detected_any (≥1 read in any sample)
# - detected_stringent (≥10 reads in ≥MIN_SAMPLES samples)
# - global_fracs (fractions by gene_type across all reads)

# 1) Universe (rows in count matrix)
universe_n = int(gene_df.shape[0])

# 2) Detected (≥1 read in any sample)
det_total = int(detected_any.sum())
det_lnc   = int(((merged['gene_type'] == 'lncRNA') & detected_any).sum())
det_pc    = int(((merged['gene_type'] == 'protein_coding') & detected_any).sum())

# 3) Robust detected (≥10 reads) — MIN_SAMPLES governs how many samples must have ≥10
rob_total = int(detected_stringent.sum())
rob_lnc   = int(((merged['gene_type'] == 'lncRNA') & detected_stringent).sum())
rob_pc    = int(((merged['gene_type'] == 'protein_coding') & detected_stringent).sum())

# 4) Global read distribution by biotype (protein_coding, lncRNA, others)
biotype_series = merged['gene_type'].where(
    merged['gene_type'].isin(['protein_coding','lncRNA']), other='outros'
)
reads_sum = merged.groupby(biotype_series)[sample_cols].sum().sum(axis=1)
reads_pct = (reads_sum / reads_sum.sum() * 100).reindex(
    ['protein_coding','lncRNA','outros']
).fillna(0).round(1)

# Save a tiny CSV for record/slide
slide_tbl = pd.DataFrame({
    'metric': ['universe_genes','detected_total','detected_lncRNA','detected_protein_coding',
               'robust_total','robust_lncRNA','robust_protein_coding',
               'reads_pct_protein_coding','reads_pct_lncRNA','reads_pct_outros'],
    'value':  [universe_n, det_total, det_lnc, det_pc,
               rob_total, rob_lnc, rob_pc,
               reads_pct['protein_coding'], reads_pct['lncRNA'], reads_pct['outros']]
})
slide_tbl.to_csv(f'{SUMMARY_EXPORT_DIR}/lncrna_slide_numbers.csv', index=False)

#### Chip-Seq

In [ ]:
# ---------- 1) Ler counts ----------
gene_df = pd.read_csv(CHIPSEQ_COUNTS_TSV, sep='\t')

# Identificar colunas de amostra
fixed_cols = {'gene_id', 'gene_name'}
sample_cols = [c for c in gene_df.columns if c not in fixed_cols]

# Remover versões do gene_id (ex.: ENSG000001.1 -> ENSG000001)
gene_df['gene_id_base'] = gene_df['gene_id'].astype(str).str.split('.').str[0]

In [ ]:
# ---------- 2) Ler GTF (somente 'gene') e extrair atributos ----------
# Definir nomes de colunas padrão para GTF e ler o arquivo ignorando comentários (#)
gtf_cols = ['seqname','source','feature','start','end','score','strand','frame','attribute']

gtf_raw = pd.read_csv(GTF_FILE, sep='\t', comment='#', header=None, names=gtf_cols)

# Extrair apenas as chaves desejadas do campo attribute
desired_keys = ['gene_id','gene_name','gene_type']
attr = gtf_raw['attribute'].apply(lambda s: parse_gtf_attributes(s, desired_keys))
attr_df = pd.json_normalize(attr)

gtf_df = pd.concat([gtf_raw.drop(columns=['attribute']), attr_df], axis=1)

# Filtrar somente a feature=gene, remove duplicatas e pega colunas essenciais
gtf_genes = (gtf_df[gtf_df['feature']=='gene']
             [['gene_id','gene_name','gene_type']]
             .drop_duplicates())

# Remove versões do gene_id para compatibilizar com a contagem
gtf_genes['gene_id_base'] = gtf_genes['gene_id'].astype(str).str.split('.').str[0]

In [ ]:
# ---------- 3) Merge counts + anotação ----------
# Join por gene_id_base e traz gene_type/gene_name do GTF
merged = (gene_df
          .merge(gtf_genes[['gene_id_base','gene_type','gene_name']],
                 on='gene_id_base', how='left', suffixes=('', '_gtf')))

merged['gene_name_final'] = merged['gene_name']
merged['gene_type'] = merged['gene_type'].fillna('unknown')

In [ ]:
# ---------- 4) Normalização TMM (edgeR) → logCPM (robusta) ----------

# 4.0  Matriz genes x amostras
counts_mat = merged[sample_cols].copy()
counts_mat.index = merged['gene_name_final'].fillna(merged['gene_id']).astype(str)

# 4.1  Saneamento p/ edgeR
# - garantir numeric, remover NA
counts_mat = counts_mat.apply(pd.to_numeric, errors='coerce').fillna(0)

# - agrupar nomes duplicados somando
counts_mat = counts_mat.groupby(counts_mat.index).sum()

# - zerar negativos e garantir inteiros
neg_n = int((counts_mat < 0).sum().sum())
if neg_n > 0:
    print(f"[WARN] {neg_n} valores negativos → definidos como 0 para edgeR.")
counts_mat = counts_mat.clip(lower=0).round().astype(np.int64)

# - remover genes com soma 0
counts_mat = counts_mat.loc[counts_mat.sum(axis=1) > 0]

# - remover amostras com biblioteca 0 (causa do 'median(f75)' NA)
lib_sizes = counts_mat.sum(axis=0)
zero_lib_cols = lib_sizes[lib_sizes == 0].index.tolist()
if zero_lib_cols:
    print(f"[WARN] Removendo {len(zero_lib_cols)} amostra(s) com soma de contagens = 0: {zero_lib_cols}")
    counts_mat = counts_mat.drop(columns=zero_lib_cols)
    # atualiza sample_cols para refletir as colunas mantidas
    sample_cols = [c for c in sample_cols if c not in zero_lib_cols]

# checagem final
assert counts_mat.shape[1] > 0, "Nenhuma amostra com contagens > 0 restou após saneamento."

# 4.2  Enviar para R e normalizar (TMM; fallback TMMwsp se necessário)
from rpy2.robjects.conversion import localconverter
from rpy2.robjects import default_converter

# ---- send counts_mat to R and run edgeR ----
with localconverter(ro.default_converter + pandas2ri.converter):
    ro.globalenv['counts_df'] = counts_mat  # auto py2rpy

ro.r("""
suppressPackageStartupMessages(library(edgeR))
dge <- DGEList(counts = as.matrix(counts_df))
dge <- calcNormFactors(dge, method = "TMM")  # or your robust fallback block
logCPM <- cpm(dge, log = TRUE, prior.count = 1)
""")

# ---- bring results back to Python (no double conversion) ----
with localconverter(ro.default_converter + pandas2ri.converter):
    logCPM_df  = ro.r('as.data.frame(logCPM)')          # -> pandas.DataFrame
    logCPM_cols = list(ro.r('colnames(logCPM)'))        # -> Python list

logCPM_df.index.name = 'gene'
logCPM_df.columns = logCPM_cols

# After you compute `logCPM_df` in R and bring it back to Python:

# 1) mean logCPM per gene (index matches counts_mat.index)
mean_logCPM_by_gene = logCPM_df.mean(axis=1)
mean_logCPM_by_gene.index.name = 'gene_key'  # just a label

# 2) Build the same key in `merged` that you used to index counts_mat
#    (you used: gene_name_final filled by gene_id)
merged['gene_key'] = merged['gene_name_final'].fillna(merged['gene_id']).astype(str)

# 3) Map the per-gene values back to *every* row in `merged`
merged['mean_logCPM'] = merged['gene_key'].map(mean_logCPM_by_gene)

# (Optional) sanity check
covered = merged['mean_logCPM'].notna().mean() * 100
print(f"[INFO] mean_logCPM mapped to merged: {covered:.1f}% rows filled")

# If you no longer need the helper key:
# merged.drop(columns='gene_key', inplace=True)

In [ ]:
# ---------- 5) Definir "detectados" ----------
# Critério 1 (liberal): pelo menos 1 read em qualquer amostra
merged['total_counts'] = merged[sample_cols].sum(axis=1)
detected_any = merged['total_counts'] > 0

# Critério 2 (stringente): >= MIN_COUNT reads em pelo menos MIN_SAMPLES amostras
detected_stringent = (merged[sample_cols] >= MIN_COUNT).sum(axis=1) >= MIN_SAMPLES

In [ ]:
# ---------- 6) Métricas globais ----------
# Resumos por critério — número de genes detectados por biotipo
summary_any = summarize_detection(detected_any, 'any_read>0')
summary_str = summarize_detection(detected_stringent, f'>={MIN_COUNT} reads in >={MIN_SAMPLES} samples')


# Frações de reads por biotipo (global, somando amostras)
reads_by_biotype = merged.groupby('gene_type')[sample_cols].sum()
global_reads = reads_by_biotype.sum(axis=1).sort_values(ascending=False)
global_total = global_reads.sum()

# Fração total por biotipo (útil para checar o peso de lncRNA vs PC)
global_fracs = (global_reads / global_total).rename('fraction')

In [ ]:
# ---------- 7) CPM clássico (não-normalizado) para compatibilidade com v0 dos gráficos ----------
# CPM médio por gene (média nas colunas de amostra)
cpm_raw, lib_sizes = make_cpm(merged, sample_cols)
merged['mean_cpm'] = cpm_raw.mean(axis=1)

In [ ]:
# ---------- 8) Tabelas auxiliares para gráficos ----------

# (a) Tabela longa para barplot: genes detectados por biotipo × critério
det_tab = pd.DataFrame([summary_any, summary_str])
det_long = det_tab.melt(id_vars=['label'],
                        value_vars=['n_lnc_detected','n_pc_detected','n_other_detected'],
                        var_name='class', value_name='n')
det_long['class'] = det_long['class'].map({'n_lnc_detected':'lncRNA',
                                           'n_pc_detected':'protein_coding',
                                           'n_other_detected':'outros'})
det_long['label'] = det_long['label'].map({'any_read>0':'Qualquer leitura (>0)',
                                          f'>={MIN_COUNT} reads in >={MIN_SAMPLES} samples':
                                          f'Robusto (≥{MIN_COUNT} reads em ≥{MIN_SAMPLES} amostras)'})


# (b) Fração por amostra (stacked 100%)
# Constrói tabela por amostra×biotipo e colapsa biotipos menos comuns em "outros"
tidy = tidy_counts_by_biotype(merged, sample_cols, 'gene_type')

main_biotypes = ['protein_coding','lncRNA']
tidy_plot = tidy.copy()
tidy_plot['biotype'] = np.where(tidy_plot['gene_type'].isin(main_biotypes),
                                tidy_plot['gene_type'], 'outros')
tidy_plot = (tidy_plot.groupby(['sample','biotype'], as_index=False)
             [['counts','total_counts']].sum())
tidy_plot['fraction'] = np.where(tidy_plot['total_counts']>0,
                                 tidy_plot['counts']/tidy_plot['total_counts'], 0.0)

# (c) Distribuição de expressão (log10 CPM) para lncRNA vs protein_coding
expr_dist = (merged[merged['gene_type'].isin(['lncRNA','protein_coding'])]
             [['gene_type','mean_logCPM']].copy())

expr_dist['log10_mean_cpm_norm'] = expr_dist['mean_logCPM']  # já é logCPM


In [ ]:
# ---------- 8) Gráficos ----------
# Configuração estética global do seaborn/matplotlib
sns.set_theme(style='whitegrid', context='talk')

# A) Genes detectados por biotipo (dois critérios)
plt.figure(figsize=(10,6))
ax = sns.barplot(data=det_long, x='label', y='n', hue='class',
                 palette={'protein_coding':'#4c78a8','lncRNA':'#f58518','outros':'#54a24b'})
ax.set_xlabel('')
ax.set_ylabel('Genes detectados')
ax.set_title('Genes detectados por biotipo')
ax.legend(title='Biotipo', frameon=False)

plt.tight_layout()

plt.savefig(f'{CHIPSEQ_LNCRNA_FIG_DIR}/A_genes_detectados_biotipo.png', dpi=300)
plt.close()

In [ ]:
# B) Fração de reads por amostra (stacked 100%)
# montar barras empilhadas 100%
samples = tidy_plot['sample'].unique()
biotypes = ['protein_coding','lncRNA','outros']
plot_mat = (tidy_plot.pivot(index='sample', columns='biotype', values='fraction')
            .reindex(columns=biotypes).fillna(0.0))
plot_mat = plot_mat.sort_index()

colors = {'protein_coding':'#4c78a8','lncRNA':'#f58518','outros':'#54a24b'}
plt.figure(figsize=(max(12, 0.5*len(samples)), 6))
bottom = np.zeros(len(plot_mat))
for bt in biotypes:
    vals = plot_mat[bt].values
    plt.bar(plot_mat.index, vals, bottom=bottom, color=colors[bt], label=bt)
    bottom += vals
plt.xticks(rotation=75, ha='right')
plt.ylim(0,1)
plt.ylabel('Fração de reads')
plt.title('Fração de reads por biotipo em cada amostra')
plt.legend(title='Biotipo', frameon=False)
plt.tight_layout()

plt.savefig(f'{CHIPSEQ_LNCRNA_FIG_DIR}/B_fracao_reads_por_amostra.png', dpi=300)
plt.close()

In [ ]:
# C) Distribuição de expressão (mean CPM) lncRNA vs protein_coding
plt.figure(figsize=(8,6))

ax = sns.violinplot(data=expr_dist, x='gene_type', y='log10_mean_cpm_norm',
                    palette={'protein_coding':'#4c78a8','lncRNA':'#f58518'}, cut=0)

# Overlay com pontos (amostra de até 2000 genes para não sobrecarregar o gráfico)
sns.stripplot(data=expr_dist.sample(min(2000, expr_dist.shape[0]), random_state=42),
              x='gene_type', y='log10_mean_cpm_norm', color='k', size=1, alpha=0.3, jitter=0.2)
ax.set_xlabel('')
ax.set_ylabel('mean logCPM (TMM)')
ax.set_title('Distribuição de expressão (TMM → logCPM)')
plt.tight_layout()

plt.savefig(f'{CHIPSEQ_LNCRNA_FIG_DIR}/C_distribuicao_expressao_log10meanCPM.png', dpi=300)
plt.close()

In [ ]:
# D) Top 20 lncRNAs (por expressão média CPM)
plt.figure(figsize=(8,10))

top_lnc = (merged[merged['gene_type']=='lncRNA']
           .sort_values('mean_logCPM', ascending=False)
           .loc[:, ['gene_id','gene_name_final','mean_logCPM']].head(20))

top_lnc_plot = top_lnc.iloc[::-1]  # inverter para barras horizontais de baixo para cima
labels = top_lnc_plot['gene_name_final'].fillna(top_lnc_plot['gene_id'])
plt.barh(labels, top_lnc_plot['mean_logCPM'], color='#f58518')
plt.xlabel('mean logCPM (TMM)')
plt.title('Top 20 lncRNAs por expressão (TMM → logCPM)')
plt.tight_layout()

plt.savefig(f'{CHIPSEQ_LNCRNA_FIG_DIR}/D_top20_lncRNAs_meanCPM.png', dpi=300)
plt.close()

In [ ]:
# === Slide helper: numbers for "Processamento: lncRNA" ===
# Uses variables already created above:
# - gene_df, merged, sample_cols
# - detected_any (≥1 read in any sample)
# - detected_stringent (≥10 reads in ≥MIN_SAMPLES samples)
# - global_fracs (fractions by gene_type across all reads)

# 1) Universe (rows in count matrix)
universe_n = int(gene_df.shape[0])

# 2) Detected (≥1 read in any sample)
det_total = int(detected_any.sum())
det_lnc   = int(((merged['gene_type'] == 'lncRNA') & detected_any).sum())
det_pc    = int(((merged['gene_type'] == 'protein_coding') & detected_any).sum())

# 3) Robust detected (≥10 reads) — MIN_SAMPLES governs how many samples must have ≥10
rob_total = int(detected_stringent.sum())
rob_lnc   = int(((merged['gene_type'] == 'lncRNA') & detected_stringent).sum())
rob_pc    = int(((merged['gene_type'] == 'protein_coding') & detected_stringent).sum())

# 4) Global read distribution by biotype (protein_coding, lncRNA, others)
biotype_series = merged['gene_type'].where(
    merged['gene_type'].isin(['protein_coding','lncRNA']), other='outros'
)
reads_sum = merged.groupby(biotype_series)[sample_cols].sum().sum(axis=1)
reads_pct = (reads_sum / reads_sum.sum() * 100).reindex(
    ['protein_coding','lncRNA','outros']
).fillna(0).round(1)

# Save a tiny CSV for record/slide
slide_tbl = pd.DataFrame({
    'metric': ['universe_genes','detected_total','detected_lncRNA','detected_protein_coding',
               'robust_total','robust_lncRNA','robust_protein_coding',
               'reads_pct_protein_coding','reads_pct_lncRNA','reads_pct_outros'],
    'value':  [universe_n, det_total, det_lnc, det_pc,
               rob_total, rob_lnc, rob_pc,
               reads_pct['protein_coding'], reads_pct['lncRNA'], reads_pct['outros']]
})
slide_tbl.to_csv(f'{SUMMARY_EXPORT_DIR}/lncrna_chipseq_slide_numbers.csv', index=False)

#### Diagrama Venn

In [ ]:
# ---- build robust ChIP set correctly (use gene_id fallback, avoid collapsing) ----
rob_chip = merged.loc[detected_stringent, ['gene_id', 'gene_name_final']].copy()
rob_chip['gene_id_base'] = rob_chip['gene_id'].astype(str).str.split('.').str[0]
# key = gene symbol if present, else Ensembl ID (base)
rob_chip['gene_key'] = np.where(
    rob_chip['gene_name_final'].notna() & (rob_chip['gene_name_final'].astype(str).str.strip() != ''),
    rob_chip['gene_name_final'].astype(str),
    rob_chip['gene_id_base'].astype(str)
)
rob_chip['gene_key_up'] = rob_chip['gene_key'].str.upper()

# Diagnostics
n_rows = rob_chip.shape[0]                                 # total robust rows (should be 131)
n_na_symbol = int(rob_chip['gene_name_final'].isna().sum())
n_unique_by_id = rob_chip['gene_id_base'].nunique()
n_unique_by_key = rob_chip['gene_key_up'].nunique()
dup_names = (rob_chip.groupby('gene_key_up').size().sort_values(ascending=False))
dup_names = dup_names[dup_names > 1]

print(f"[ChIP robust] rows={n_rows}, unique_ids={n_unique_by_id}, "
      f"unique_keys={n_unique_by_key}, na_symbols={n_na_symbol}")
if not dup_names.empty:
    print("[ChIP robust] note: multiple Ensembl IDs share the same symbol; top collisions:")
    print(dup_names.head(10))

# FINAL robust set for Venn (size should now match 131 if that’s your robust count)
chip_robust_set = set(rob_chip['gene_key_up'])

A = chip_robust_set                  # ChIP robust (should be 131)
B = set(robust_lnc_set)              # RNA-seq lncRNA robust (uppercase)

a_only = len(A - B)
b_only = len(B - A)
ab     = len(A & B)
na, nb = len(A), len(B)

imbalance = min(na, nb) / max(na, nb) if max(na, nb) else 1.0
use_unweighted = (imbalance < 0.4)

plt.figure(figsize=(8,6))
v = (venn2_unweighted if use_unweighted else venn2)(
    subsets=(a_only, b_only, ab),
    set_labels=(f"ChIP-seq robust (n={na:,})", f"RNA-seq lncRNA robust (n={nb:,})")
)

for pid in ('10','01','11'):
    if v.get_patch_by_id(pid):
        v.get_patch_by_id(pid).set_alpha(0.65)
        v.get_patch_by_id(pid).set_edgecolor('#333'); v.get_patch_by_id(pid).set_linewidth(1.2)
if v.get_patch_by_id('10'): v.get_patch_by_id('10').set_color('#4c78a8')
if v.get_patch_by_id('01'): v.get_patch_by_id('01').set_color('#f58518')
if v.get_patch_by_id('11'): v.get_patch_by_id('11').set_color('#9ecae1')

def _fmt(n): return f"{n:,}"
def _pct(n, d): return f"{(n/d*100):.1f}%" if d else "0.0%"

if v.get_label_by_id('10'): v.get_label_by_id('10').set_text(f"{_fmt(a_only)}\n({_pct(a_only, na)})")
if v.get_label_by_id('01'): v.get_label_by_id('01').set_text(f"{_fmt(b_only)}\n({_pct(b_only, nb)})")
if v.get_label_by_id('11'): v.get_label_by_id('11').set_text(f"{_fmt(ab)}")

for lid in ('10','01','11'):
    lbl = v.get_label_by_id(lid)
    if lbl: lbl.set_fontsize(12)
for s in v.set_labels:
    if s: s.set_fontsize(12)

plt.title(f"Sobreposição: ChIP-seq (robusto) × RNA-seq lncRNA (robusto)", fontsize=13, pad=12)
plt.tight_layout()
plt.savefig(f"{VENN_FIG_DIR}/venn_chip_vs_rnaseq_lnc_robust_readable.png", dpi=300); plt.close()

# Save the overlap table
pd.DataFrame(sorted(A & B), columns=['gene']).to_csv(f"{SUMMARY_EXPORT_DIR}/venn_overlap_genes.csv", index=False)

### Processamento: WGCNA

#### O que esta seção faz
--------------------
1) Lê a anotação (GTF) e constrói um mapa gene_name→biotype.
2) Lê nodes/edges (formato Cytoscape) do WGCNA e anota cada nó com biotipo e módulo.
3) Estima um *cutoff* automático de peso (percentil 95) para manter ~5% das arestas
(apenas para depuração), **e aplica um cutoff calculado (WEIGHT_CUTOFF_AUTO)** na filtragem efetiva.
4) Aplica um filtro biológico adicional: se houver lncRNAs em uma aresta, eles devem
pertencer ao conjunto `robust_lnc_set` (derivado da parte de lncRNA acima; isto
reduz falsos positivos de lncRNAs com baixa/instável detecção).
5) Exporta sub-redes compatíveis com Cytoscape (edges/nodes filtrados) e estatísticas
de redução e estrutura (para slides e QC).
6) Gera *exports por módulo* (nodes/edges) coloridos, marcando hubs e genes-alvo.

In [ ]:
# ---------- 1) Ler GTF e fazer mapa gene_name -> biotipo ----------
# Usa a função utilitária que prioriza lncRNA > protein_coding quando houver ambiguidade
gtf = load_gtf_gene_biotype(GTF_FILE)
name2type = dict(zip(gtf['gene_name'].str.upper(), gtf['gene_type']))

In [ ]:
# ---------- 2) Ler nodes/edges (formato Cytoscape) ----------
nodes = pd.read_csv(NODE_FILE, sep='\t')
if 'nodeAttr[nodesPresent, ]' in nodes.columns:
    nodes = nodes.rename(columns={'nodeAttr[nodesPresent, ]':'module'})
assert {'nodeName','module'}.issubset(nodes.columns), "Arquivo de nodes precisa ter colunas 'nodeName' e 'module'."

# Normalizar nomes e adicionar biotipos
nodes['gene_name']    = nodes['nodeName'].astype(str)
nodes['gene_name_up'] = nodes['gene_name'].str.upper()
nodes['biotype']      = nodes['gene_name_up'].map(name2type).fillna('other')

node2bio    = dict(zip(nodes['nodeName'], nodes['biotype']))
node2module = dict(zip(nodes['nodeName'], nodes['module']))

In [ ]:
# ---------- 3) Encontrar automaticamente o corte que remove 95% das arestas ----------
# **Objetivo:** saber qual peso mantém ~5% das arestas. Isso é útil para calibrar o
# cutoff fixo. A leitura é em *chunks* para economizar memória.

# Histograma de alta resolução para calcular o percentil 95 de 'weight'
hist_bins   = np.linspace(0, 1, 400+1)      # 400 bins p/ boa resolução
hist_counts = np.zeros(len(hist_bins)-1, dtype=np.int64)
total_edges = 0

# Lendo em chunks de 2MI
for ch in pd.read_csv(EDGE_FILE, sep='\t', usecols=['fromNode','toNode','weight'], chunksize=2_000_000):
    ch = ch.dropna()
    w  = ch['weight'].astype(float).values
    h,_ = np.histogram(w, bins=hist_bins)
    hist_counts += h
    total_edges += len(ch)
    del ch
    gc.collect()

# Cálculo do percentil 95 a partir da CDF aproximada do histograma
cum = hist_counts.cumsum() / max(hist_counts.sum(), 1)
idx = np.searchsorted(cum, PERCENTILE_THRESHOLD)

WEIGHT_CUTOFF_AUTO = float(hist_bins[min(idx, len(hist_bins)-1)])

print(f"[INFO] 95th percentile cutoff = {WEIGHT_CUTOFF_AUTO:.6f}  (mantendo ~5% das arestas)")

In [ ]:
# ---------- 4) Filtrar arestas (segunda passada) e salvar o que restou ----------
# Saída: arquivo TSV com arestas filtradas para inspeção no Cytoscape.
filtered_edges_path = f'{WGCNA_FILTERED_EXPORT_DIR}/CytoscapeInput-edges.filtered.top5pct.robust.tsv'
with open(filtered_edges_path, 'w') as f_out:
    f_out.write('fromNode\ttoNode\tweight\n')

# Função auxiliar: garante que lncRNAs envolvidos sejam "robustos"
def _keep_edge(u, v):
    bu, bv = node2bio.get(u, 'other'), node2bio.get(v, 'other')
    if bu == 'lncRNA' and u.upper() not in robust_lnc_set:  # robust_lnc_set vem da parte lncRNA
        return False
    if bv == 'lncRNA' and v.upper() not in robust_lnc_set:
        return False
    return True

nodes_seen = set() # nós que aparecem em pelo menos uma aresta mantida
E_kept = 0

for ch in pd.read_csv(EDGE_FILE, sep='\t', usecols=['fromNode','toNode','weight'], chunksize=2_000_000):
    # 4a) Filtro por peso
    ch = ch[ch['weight'].astype(float) >= WEIGHT_CUTOFF_AUTO].copy()
    if ch.empty:
        continue

    # 4b) Filtro biológico — remove arestas que incluem lncRNA não-robusto
    mask = [ _keep_edge(u, v) for u, v in ch[['fromNode','toNode']].itertuples(index=False, name=None) ]
    ch = ch.loc[mask]
    if ch.empty:
        continue

    # 4c) Export incremental (modo append) para evitar manter tudo em RAM
    ch[['fromNode','toNode','weight']].to_csv(filtered_edges_path, sep='\t', index=False, header=False, mode='a')

    # 4d) Atualiza conjunto de nós presentes
    nodes_seen.update(ch['fromNode'].astype(str))
    nodes_seen.update(ch['toNode'].astype(str))
    E_kept += len(ch)
    del ch

print(f"[INFO] Arestas mantidas após ambos filtros: {E_kept:,} ({E_kept/total_edges*100:.2f}%)")

In [ ]:
# ---------- 5) Salvar nodes filtrados (apenas os presentes nas edges remanescentes) ----------
# Gera um nodes.tsv reduzido (facilita carregar no Cytoscape apenas o subgrafo relevante).

nodes_filt = nodes[nodes['nodeName'].isin(nodes_seen)].copy()
nodes_filt['is_robust_lnc'] = (nodes_filt['biotype'].eq('lncRNA') &
                               nodes_filt['gene_name_up'].isin(robust_lnc_set))

# Removemos lncRNAs não-robustos que, por acaso, tenham aparecido (sanity check)
nodes_filt = nodes_filt[(nodes_filt['biotype']!='lncRNA') | (nodes_filt['is_robust_lnc'])].copy()

nodes_filt_path = f'{WGCNA_FILTERED_EXPORT_DIR}/CytoscapeInput-nodes.filtered.top5pct.robust.tsv'
nodes_filt[['nodeName','gene_name','module','biotype','is_robust_lnc']].to_csv(nodes_filt_path, sep='\t', index=False)

In [ ]:
# ------------------------
# 6) Distribuição das arestas (Original vs Filtrado) p/ o slide
# ------------------------
# Gera uma tabela com contagens de arestas e número de nós por biotipo,
# antes e depois do filtro. Útil para comunicar a magnitude da redução

def _edge_counts(edge_path, filtered=False):
    stats = dict(E_total=0, E_lnc=0, E_gene=0, lnc_nodes=set(), pc_nodes=set())
    node_bio_map = node2bio if not filtered else dict(zip(nodes_filt['nodeName'], nodes_filt['biotype']))
    for ch in pd.read_csv(edge_path, sep='\t', chunksize=2_000_000):
        s_type = ch['fromNode'].map(node_bio_map).fillna('other')
        t_type = ch['toNode'].map(node_bio_map).fillna('other')
        stats['E_total'] += len(ch)
        stats['E_lnc']   += int((s_type.eq('lncRNA') | t_type.eq('lncRNA')).sum())
        stats['E_gene']  += int((s_type.eq('protein_coding') & t_type.eq('protein_coding')).sum())
        stats['lnc_nodes'].update(ch.loc[s_type.eq('lncRNA'),'fromNode'])
        stats['lnc_nodes'].update(ch.loc[t_type.eq('lncRNA'),'toNode'])
        stats['pc_nodes'].update(ch.loc[s_type.eq('protein_coding'),'fromNode'])
        stats['pc_nodes'].update(ch.loc[t_type.eq('protein_coding'),'toNode'])
    return stats

# criar snapshot de edges originais (3 colunas) só para sumarizar
_tmp_all_edges = f'{SUMMARY_EXPORT_DIR}/__all_edges_tmp.tsv'
if not os.path.exists(_tmp_all_edges):
    with open(_tmp_all_edges, 'w') as f: f.write('fromNode\ttoNode\tweight\n')
    for ch in pd.read_csv(EDGE_FILE, sep='\t', usecols=['fromNode','toNode','weight'], chunksize=2_000_000):
        ch.to_csv(_tmp_all_edges, sep='\t', index=False, header=False, mode='a')

s_orig = _edge_counts(_tmp_all_edges, filtered=False)
s_filt = _edge_counts(filtered_edges_path, filtered=True)

edges_tbl = pd.DataFrame([
    ('Total de arestas', s_orig['E_total'], s_filt['E_total']),
    ('Nós-lncRNA',       len(s_orig['lnc_nodes']),  len(s_filt['lnc_nodes'])),
    ('Interações lncRNA',s_orig['E_lnc'],          s_filt['E_lnc']),
    ('Nós-Genes',        len(s_orig['pc_nodes']),  len(s_filt['pc_nodes'])),
    ('Interações Genes', s_orig['E_gene'],         s_filt['E_gene']),
], columns=['Métrica','Original','Filtrado'])

edges_tbl['Redução'] = (1 - (edges_tbl['Filtrado']/edges_tbl['Original'])).map(lambda x: f'{x*100:.0f}%')
edges_tbl.to_csv(f'{SUMMARY_EXPORT_DIR}/edges_summary.top5pct_robust.csv', index=False)

In [ ]:
# ------------------------
# 7) Estrutura da rede (filtrado) p/ o slide
# ------------------------
# Calculamos grau por nó, densidade, número de componentes e tamanho do maior componente
# usando uma DSU (Union-Find) simples, sem dependências externas.

deg = {}
parent = {}
def _find(x):
    parent.setdefault(x, x)
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x
def _union(a,b):
    ra, rb = _find(a), _find(b)
    if ra != rb: parent[rb] = ra

E = 0
for ch in pd.read_csv(filtered_edges_path, sep='\t', chunksize=2_000_000):
    E += len(ch)
    for u,v in ch[['fromNode','toNode']].itertuples(index=False):
        deg[u] = deg.get(u,0)+1; deg[v] = deg.get(v,0)+1
        _union(u,v)

N = len(nodes_filt)
avg_degree = (2*E)/N if N>0 else np.nan
density    = (2*E)/(N*(N-1)) if N>1 else np.nan
comp_size  = {}

for n in nodes_filt['nodeName']:
    r = _find(n); comp_size[r] = comp_size.get(r,0)+1

n_components    = len(comp_size)
giant_component = max(comp_size.values()) if comp_size else 0

net_metrics = pd.DataFrame({
    'Métrica': ['Nós (N)','Arestas (E)','Grau médio (2E/N)','Densidade (2E/N(N-1))','# Componentes','Maior componente (nós)'],
    'Valor':   [N, E, avg_degree, density, n_components, giant_component]
})

net_metrics.to_csv(f'{SUMMARY_EXPORT_DIR}/network_structure_metrics.top5pct_robust.csv', index=False)

In [ ]:
# ------------------------
# 8) Módulos — barras coloridas pela cor do módulo
# ------------------------

# Gera uma figura (PNG) com o tamanho de cada módulo e a porcentagem de lncRNA.
WGCNA_COLOR_HEX = {
    'black':'#000000','blue':'#1f77b4','brown':'#8B4513','cyan':'#00FFFF','darkgreen':'#006400',
    'darkgrey':'#555555','darkmagenta':'#8B008B','darkolivegreen':'#556B2F','darkorange':'#FF8C00',
    'darkred':'#8B0000','darkturquoise':'#00CED1','green':'#2ca02c','greenyellow':'#ADFF2F',
    'grey60':'#999999','ivory':'#FFFFF0','lightcyan':'#E0FFFF','lightcyan1':'#E0FFFF',
    'lightgreen':'#90EE90','lightsteelblue1':'#CAE1FF','lightyellow':'#FFFFE0','magenta':'#FF00FF',
    'mediumpurple3':'#8968CD','midnightblue':'#191970','orange':'#FFA500','orangered4':'#8B2500',
    'paleturquoise':'#AFEEEE','pink':'#FFC0CB','plum1':'#FFBBFF','purple':'#800080','royalblue':'#4169E1',
    'saddlebrown':'#8B4513','salmon':'#FA8072','sienna3':'#CD6839','skyblue':'#87CEEB','skyblue3':'#6CA6CD',
    'steelblue':'#4682B4','tan':'#D2B48C','violet':'#EE82EE','white':'#FFFFFF','yellowgreen':'#9ACD32'
}

nodes_mod = nodes_filt.copy()
sizes     = nodes_mod.groupby('module').size().rename('n')
lnc_sizes = nodes_mod[nodes_mod['biotype']=='lncRNA'].groupby('module').size().rename('lnc')
mod_stats = pd.concat([sizes, lnc_sizes], axis=1).fillna(0).reset_index()
mod_stats['lnc_pct'] = (mod_stats['lnc']/mod_stats['n']*100)
mod_stats = mod_stats.sort_values('n', ascending=False)
mod_stats.to_csv(f'{MODULE_EXPORT_DIR}/modules_sizes_lnc_top5pct_robust.csv', index=False)

plt.figure(figsize=(max(10,0.5*len(mod_stats)),6))
bar_colors = [WGCNA_COLOR_HEX.get(m, '#6baed6') for m in mod_stats['module']]
ax = sns.barplot(data=mod_stats, x='module', y='n', palette=bar_colors)
for i,(m,n,p) in enumerate(zip(mod_stats['module'], mod_stats['n'], mod_stats['lnc_pct'])):
    ax.text(i, n+max(mod_stats['n'])*0.01, f"{p:.1f}% lnc", ha='center', va='bottom', fontsize=10)
plt.xticks(rotation=75, ha='right')
plt.ylabel('Nós no módulo')
plt.xlabel('Módulo (cor)')
plt.title('WGCNA: Módulos — tamanho e % lncRNA (top 5% + lnc robusto)')
plt.tight_layout()

plt.savefig(f'{WGCNA_FIG_DIR}/modules_sizes_lnc_colored.png', dpi=300)
plt.close()

In [ ]:
# ------------------------
# 8) Hubs (usar arquivo fornecido, sem API) e marcar nos exports
# ------------------------
# Ler hubs.csv e limpar cores com quebras/aspas
hubs = pd.read_csv(HUBS_CSV, sep=None, engine='python')
hubs.columns = [c.strip().lower() for c in hubs.columns]

# normalizar colunas esperadas: module, gene, function
if 'module' not in hubs.columns or 'gene' not in hubs.columns:
    raise ValueError(f"hubs.csv deve conter colunas 'module' e 'gene'. Encontrado: {list(hubs.columns)}")
if 'function' not in hubs.columns:
    hubs['function'] = ''

hubs['module'] = hubs['module'].astype(str).str.strip().str.replace(r'["\']','', regex=True).str.replace(r'\s+','', regex=True)
hubs['gene']   = hubs['gene'].astype(str).str.strip()
hubs['gene_up'] = hubs['gene'].str.upper()
hubs_map = dict(zip(zip(hubs['module'], hubs['gene_up']), hubs['function']))

In [ ]:
# ------------------------
# 9) Exports por módulo p/ Cytoscape (um par nodes/edges por módulo), destacando hubs e genes-alvo
# ------------------------
TARGET_GENES = {'GATA6','GATA6-AS1'}  # ajuste se quisermos destacar outros
mods = sorted(nodes_mod['module'].unique())
mod_nodesets = {m: set(nodes_mod.loc[nodes_mod['module']==m, 'nodeName']) for m in mods}

# Cabeçalhos — criamos arquivos destinos vazios (modo append virá depois)
for m in mods:
    with open(f'{MODULE_EXPORT_DIR}/{m}_nodes.tsv','w') as fn:
        fn.write('nodeName\tgene_name\tmodule\tbiotype\tis_robust_lnc\tis_hub\thub_function\thighlight\n')
    with open(f'{MODULE_EXPORT_DIR}/{m}_edges.tsv','w') as fe:
        fe.write('fromNode\ttoNode\tweight\n')

# (9a) Nodes por módulo com marcações de hub/lnc robusto/targets
for m in mods:
    sub = nodes_mod[nodes_mod['module']==m].copy()
    sub['is_hub'] = sub['gene_name'].str.upper().apply(lambda g: 1 if (m, g) in hubs_map else 0)
    sub['hub_function'] = sub.apply(lambda r: hubs_map.get((m, r['gene_name'].upper()), ''), axis=1)

    tgt_mask = sub['gene_name'].str.upper().isin(TARGET_GENES)
    lnc_mask = sub['biotype'].eq('lncRNA') & sub['is_robust_lnc'].fillna(True).astype(bool)
    sub['highlight'] = np.select([lnc_mask, tgt_mask], [2, 1], default=0).astype(int)
    sub[['nodeName','gene_name','module','biotype','is_robust_lnc','is_hub','hub_function','highlight']].to_csv(
        f'{MODULE_EXPORT_DIR}/{m}_nodes.tsv', sep='\t', index=False, mode='a', header=False
    )

# (9b) Edges intramodulares por módulo — iteramos os *chunks* das arestas filtradas
for ch in pd.read_csv(filtered_edges_path, sep='\t', chunksize=2_000_000):
    for m in mods:
        s = mod_nodesets[m]
        chm = ch[ch['fromNode'].isin(s) & ch['toNode'].isin(s)]
        if not chm.empty:
            chm.to_csv(f'{MODULE_EXPORT_DIR}/{m}_edges.tsv', sep='\t', index=False, mode='a', header=False)

# (9c) Verificação simples: hubs do CSV estão presentes no subgrafo exportado?
hubs_present = []
for m in mods:
    in_mod = nodes_mod.loc[nodes_mod['module']==m, 'gene_name'].str.upper()
    hl = hubs[hubs['module']==m]
    if not hl.empty:
        gene_up = hl['gene_up'].iloc[0]
        hubs_present.append({
            'module': m,
            'hub_gene': hl['gene'].iloc[0],
            'present_in_network': bool((in_mod==gene_up).any()),
            'function': hl['function'].iloc[0]
        })
pd.DataFrame(hubs_present).to_csv(f'{MODULE_EXPORT_DIR}/hubs_in_network_check.csv', index=False)

In [ ]:
# Focando no Brown
brown_nodes = pd.read_csv(f'{MODULE_EXPORT_DIR}/brown_nodes.tsv', sep="\t")
brown_edges = pd.read_csv(f'{MODULE_EXPORT_DIR}/brown_edges.tsv', sep="\t")

# FIltering lncRNA or GATA6
nodes_lncRNA = brown_nodes[(brown_nodes['biotype'] == 'lncRNA') | (brown_nodes['nodeName'].isin(['GATA6', 'GATA6-AS1']))]

# List of notes
lst = nodes_lncRNA['nodeName'].tolist()

# Filtering only the nodes above
brown_edges = brown_edges[(brown_edges['fromNode'].isin(lst)) | (brown_edges['toNode'].isin(lst))]

# List of nodes on Edges after filtering
lst1 = brown_edges['fromNode'].tolist()
lst2 = brown_edges['toNode'].tolist()

# Filter nodes file with those edges
brown_nodes = brown_nodes[(brown_nodes['nodeName'].isin(lst1)) | (brown_nodes['nodeName'].isin(lst2))]

# Save csv
brown_nodes.to_csv(f'{MODULE_EXPORT_DIR}/brown_nodes_filtered.tsv', sep='\t', index=False)
brown_edges.to_csv(f'{MODULE_EXPORT_DIR}/brown_edges_filtered.tsv', sep='\t', index=False)

### Processamento: Terceira entrega

#### FILTROS DE REDE (a–g)

In [ ]:
# =======================================================
# Escolha da fonte da rede base (NORMALIZADA) e ajustando colunas
# =======================================================

# 1) Carregar
nodes_raw = pd.read_csv(NORMALIZED_NODE_FILE, sep="\t")
edges     = pd.read_csv(NORMALIZED_EDGE_FILE, sep="\t")

# 2) Harmonizar nomes de colunas mínimos esperados pelo notebook
#    - nodeName
#    - module (cor do módulo)
#    - gene_name
colmap = {}
if "name" in nodes_raw.columns and "nodeName" not in nodes_raw.columns:
    colmap["name"] = "nodeName"
if "moduleColor" in nodes_raw.columns and "module" not in nodes_raw.columns:
    colmap["moduleColor"] = "module"
if "color" in nodes_raw.columns and "module" not in nodes_raw.columns:
    colmap["color"] = "module"
if "group" in nodes_raw.columns and "module" not in nodes_raw.columns:
    colmap["group"] = "module"

# aplica renomeações detectadas
nodes = nodes_raw.rename(columns=colmap).copy()

# 3) Garantir colunas mínimas
# gene_name: se não existir, use o próprio nodeName (nos arquivos Cytoscape, 'nodeName' costuma ser o símbolo)
if "gene_name" not in nodes.columns:
    if "nodeName" not in nodes.columns:
        raise KeyError(f"'nodeName' ausente em {NORMALIZED_NODE_FILE}. Colunas: {list(nodes.columns)}")
    nodes["gene_name"] = nodes["nodeName"].astype(str)

# module: se não existir, tente a coluna do WGCNA exportada pelo R: 'nodeAttr[nodesPresent, ]'
if "module" not in nodes.columns:
    if "nodeAttr[nodesPresent, ]" in nodes.columns:
        nodes = nodes.rename(columns={"nodeAttr[nodesPresent, ]": "module"})
    else:
        # se ainda não existir, preencha como 'grey' (WGCNA convencional para "sem módulo")
        nodes["module"] = "grey"

# biotype: se não existir, anotar via GTF (GENCODE v46)
if "biotype" not in nodes.columns:
    # cria um mapa SYMBOL_UPPER -> gene_type a partir do GTF
    gtf_cols = ['seqname','source','feature','start','end','score','strand','frame','attribute']
    gtf_raw  = pd.read_csv(GTF_FILE, sep="\t", comment="#", header=None, names=gtf_cols)
    attr = gtf_raw[gtf_raw['feature']=="gene"]["attribute"].apply(
        lambda s: parse_gtf_attributes(s, ["gene_id","gene_name","gene_type"])
    )
    attr = pd.json_normalize(attr).dropna(subset=["gene_name"])
    name2type = dict(zip(attr["gene_name"].astype(str).str.upper(), attr["gene_type"].astype(str)))
    nodes["biotype"] = nodes["gene_name"].astype(str).str.upper().map(name2type).fillna("unknown")

# 4) Harmonizar EDGES
edges = edges.copy()
# nomes das colunas de origem/destino
if "fromNode" not in edges.columns or "toNode" not in edges.columns:
    cand_map = {}
    if "from" in edges.columns and "fromNode" not in edges.columns:
        cand_map["from"] = "fromNode"
    if "to" in edges.columns and "toNode" not in edges.columns:
        cand_map["to"] = "toNode"
    if "source" in edges.columns and "fromNode" not in edges.columns:
        cand_map["source"] = "fromNode"
    if "target" in edges.columns and "toNode" not in edges.columns:
        cand_map["target"] = "toNode"
    if cand_map:
        edges = edges.rename(columns=cand_map)
    # checagem final
    if "fromNode" not in edges.columns or "toNode" not in edges.columns:
        raise KeyError(f"Não encontrei colunas de aresta ('fromNode'/'toNode') em {NORMALIZED_EDGE_FILE}. Colunas: {list(edges.columns)}")

# coluna de peso → padronizar para 'weight'
if "weight" not in edges.columns:
    for cand in ["weight_norm","w_norm","corr","score","adjacency","similarity"]:
        if cand in edges.columns:
            edges = edges.rename(columns={cand: "weight"})
            break
    else:
        # se não houver peso explícito, crie um peso unitário (permite seguir com filtros topológicos)
        edges["weight"] = 1.0

# 5) Atribuir aos objetos padrão do notebook
base_nodes = nodes
base_edges = edges

# 6) Preparar mapeamentos convenientes
symb_up = dict(zip(base_nodes['nodeName'],
                   base_nodes['gene_name'].fillna('').astype(str).str.upper()))
biotype = dict(zip(base_nodes['nodeName'], base_nodes['biotype']))

# 7) Recalcular ALWAYS_NODES após termos 'base_nodes' pronto
#    (EXTRA_SYMBOLS_ALWAYS deve existir; ex.: EXTRA_SYMBOLS_ALWAYS = ['GATA6','GATA6-AS1'])
ALWAYS_NODES = symbols_to_nodes(EXTRA_SYMBOLS_ALWAYS, base_nodes)

# 8) Checagens rápidas
missing_cols_nodes = {'nodeName','gene_name','module','biotype'} - set(base_nodes.columns)
missing_cols_edges = {'fromNode','toNode','weight'} - set(base_edges.columns)
if missing_cols_nodes:
    print("[WARN] base_nodes sem colunas esperadas:", missing_cols_nodes)
if missing_cols_edges:
    print("[WARN] base_edges sem colunas esperadas:", missing_cols_edges)

print("[OK] Rede base (NORMALIZADA) harmonizada:")
print(f" - Nós (base_nodes): {base_nodes.shape[0]}  | colunas: {list(base_nodes.columns)[:8]} ...")
print(f" - Arestas (base_edges): {base_edges.shape[0]} | colunas: {list(base_edges.columns)[:8]} ...")
print(f" - ALWAYS_NODES resolvidos ({len(ALWAYS_NODES)}):", sorted(list(ALWAYS_NODES))[:6], "...")

In [ ]:
# ===============================
# (a) Top-5 por nó
# ===============================}
edges_a = topk_per_node(base_edges, k=5)
export_cytoscape(edges_a, base_nodes, f"{FILTER_EXPORT_DIR}/a_top5_per_node", extra_nodes=ALWAYS_NODES)
met_a = save_metrics('a_top5_per_node', edges_a, base_nodes, extra_nodes=ALWAYS_NODES)

In [ ]:
# ===============================
# (b) 95% - Versão original, mantendo a referência
# ===============================
edges_b = base_edges.copy()

export_cytoscape(edges_b, base_nodes, f"{FILTER_EXPORT_DIR}/b_top5pct_weight", extra_nodes=ALWAYS_NODES)
met_b = save_metrics('b_top5pct_weight', edges_b, base_nodes, extra_nodes=ALWAYS_NODES)

In [ ]:
# ===============================
# (b.1) 99% - Filtrado para ser carregado no cytoscape
# ===============================


#Encontrar automaticamente o corte que remove 99% das arestas ----------
# Objetivo: saber qual peso mantém ~1% das arestas. Isso é útil para calibrar o cutoff fixo.

# Histograma de alta resolução para calcular o percentil 95 de 'weight'
hist_bins   = np.linspace(0, 1, 400+1)      # 400 bins p/ boa resolução
hist_counts = np.zeros(len(hist_bins)-1, dtype=np.int64)
total_edges = 0

# Lendo em chunks de 2MI
for ch in pd.read_csv(NORMALIZED_EDGE_FILE, sep='\t', usecols=['fromNode','toNode','weight'], chunksize=2_000_000):
    ch = ch.dropna()
    w  = ch['weight'].astype(float).values
    h,_ = np.histogram(w, bins=hist_bins)
    hist_counts += h
    total_edges += len(ch)
    del ch
    gc.collect()

# Cálculo do percentil 95 a partir da CDF aproximada do histograma
cum = hist_counts.cumsum() / max(hist_counts.sum(), 1)
idx = np.searchsorted(cum, PERCENTILE_THRESHOLD_B_FILTER)

WEIGHT_CUTOFF_AUTO = float(hist_bins[min(idx, len(hist_bins)-1)])

print(f"[INFO] 99th percentile cutoff = {WEIGHT_CUTOFF_AUTO:.6f}  (mantendo ~1% das arestas para o filtro B)")



edges_b = base_edges[base_edges['weight'].astype(float) >= WEIGHT_CUTOFF_AUTO].copy()

export_cytoscape(edges_b, base_nodes, f"{FILTER_EXPORT_DIR}/b_top1pct_weight", extra_nodes=ALWAYS_NODES)
met_b_1 = save_metrics('b_top1pct_weight', edges_b, base_nodes, extra_nodes=ALWAYS_NODES)

In [ ]:
# ===============================
# (c) Vizinhos (1-hop) de GATA6/GATA6-AS1 (subgrafo induzido) - Versão original, mantendo a referência
# ===============================
SEEDS = {'GATA6','GATA6-AS1'}
seed_nodes = {n for n,s in symb_up.items() if s in SEEDS}
edges_c = base_edges[(base_edges['fromNode'].isin(seed_nodes)) | (base_edges['toNode'].isin(seed_nodes))].copy()
keep_nodes_c = set(edges_c['fromNode']) | set(edges_c['toNode'])
edges_c = base_edges[base_edges['fromNode'].isin(keep_nodes_c) & base_edges['toNode'].isin(keep_nodes_c)]
export_cytoscape(edges_c, base_nodes, f"{FILTER_EXPORT_DIR}/c_neighbors_of_GATA6", extra_nodes=ALWAYS_NODES)

met_c = save_metrics('c_neighbors_of_GATA6', edges_c, base_nodes, extra_nodes=ALWAYS_NODES)

In [ ]:
# ===============================
# (c).1 Vizinhos (1-hop) de GATA6/GATA6-AS1 (subgrafo induzido) + Top-K por nó
#     - Sementes: {GATA6, GATA6-AS1}
#     - Nós do subgrafo: sementes ∪ vizinhos diretos
#     - Arestas do subgrafo: apenas entre nós desse conjunto (inclui vizinho↔vizinho)
#     - Redução: Top-K arestas de maior peso por nó (K=5)
#     - ALWAYS_NODES são adicionados no export (podem ficar isolados)
# ===============================

K_TOP = 5
SEEDS = {'GATA6', 'GATA6-AS1'}

# 1) nodeNames das sementes na rede base (via símbolo em caixa alta)
seed_nodes = {n for n, sym in symb_up.items() if sym in SEEDS}
if not seed_nodes:
    print("[c-TopK] AVISO: não encontrei sementes na rede base; seguindo com ALWAYS_NODES no export.")
    seed_nodes = set()

# 2) Arestas incidentes às sementes
incident = base_edges[
    base_edges['fromNode'].isin(seed_nodes) | base_edges['toNode'].isin(seed_nodes)
].copy()

# 3) Vizinhos diretos = o outro extremo das arestas seed↔X
direct_neighbors = set(
    np.where(incident['fromNode'].isin(seed_nodes), incident['toNode'], incident['fromNode'])
)

# 4) Conjunto de nós do subgrafo induzido
keep_nodes_c = seed_nodes | direct_neighbors

# 5) Subgrafo induzido: somente arestas cujos dois nós ∈ keep_nodes_c
edges_c_full = base_edges[
    base_edges['fromNode'].isin(keep_nodes_c) &
    base_edges['toNode'].isin(keep_nodes_c)
].copy()

# 6) Top-K por nó (dentro do subgrafo induzido)
if not edges_c_full.empty:
    # ordenar por peso desc para usar .head(K)
    edges_sorted = edges_c_full.sort_values('weight', ascending=False)

    # Top-K pelas duas "visões" (garante simetria): por fromNode e por toNode
    topk_from = edges_sorted.groupby('fromNode', as_index=False, group_keys=False).head(K_TOP)
    topk_to   = edges_sorted.groupby('toNode',   as_index=False, group_keys=False).head(K_TOP)

    # União + deduplicação por par não-direcionado
    e_topk = pd.concat([topk_from, topk_to], ignore_index=True)
    pair = np.where(e_topk['fromNode'] < e_topk['toNode'],
                    e_topk['fromNode'] + '||' + e_topk['toNode'],
                    e_topk['toNode']   + '||' + e_topk['fromNode'])
    e_topk = (e_topk.assign(_pair=pair)
                    .sort_values('weight', ascending=False)
                    .drop_duplicates('_pair')
                    .drop(columns=['_pair']))
else:
    e_topk = edges_c_full.copy()

# 7) Exportar (Cytoscape) com ALWAYS_NODES garantidos no arquivo de nós
out_prefix = f"{FILTER_EXPORT_DIR}/c_neighbors_of_GATA6_GATA6AS1_topK_{K_TOP}"
export_cytoscape(e_topk, base_nodes, out_prefix, extra_nodes=ALWAYS_NODES)

# 8) Métricas e logs
met_c_1 = save_metrics(f'c_neighbors_of_GATA6_GATA6AS1_topK_{K_TOP}', e_topk, base_nodes, extra_nodes=ALWAYS_NODES)

In [ ]:
# ===============================
# (D) DEG estrito — padj < 0.05 (|log2FC| ≥ 0.0)
#   - União dos DEGs dos dois contrastes
#   - Mapeia por símbolo e fallback por ENSG
#   - Mantém somente arestas DEG↔DEG
#   - Exporta para Cytoscape
# ===============================
DIFFEXP_DIR = f"{SOURCE_FOLDER}/data/processed/diff_expression/tables/differential"
CONTRASTS_D = ["siGATA6_vs_siCtl_Veh", "siGATA6_vs_siCtl_DEX"]
Path(FILTER_EXPORT_DIR).mkdir(parents=True, exist_ok=True)

# --- Mapa ENSG_base -> símbolo/biotipo (GENCODE v46) ---
_gtf_cols = ['seqname','source','feature','start','end','score','strand','frame','attribute']
if '_gtf_raw' not in globals():
    _gtf_raw = pd.read_csv(GTF_FILE, sep="\t", comment="#", header=None, names=_gtf_cols)

_ga = _gtf_raw[_gtf_raw['feature']=='gene']['attribute'].apply(
    lambda s: parse_gtf_attributes(s, ['gene_id','gene_name','gene_type'])
)
_ga = pd.json_normalize(_ga).dropna(subset=['gene_id'])
_ga['gene_id_base'] = _ga['gene_id'].astype(str).str.split('.').str[0]
ENSG2SYM  = dict(zip(_ga['gene_id_base'], _ga['gene_name'].astype(str)))
ENSG2TYPE = dict(zip(_ga['gene_id_base'], _ga['gene_type'].astype(str)))

def _carrega_deg_anotado(contrast: str) -> pd.DataFrame:
    """
    Lê o TSV do nf-core/DESeq2, anota com símbolo/biotipo e define 'pass'
    (|log2FC|>=0.0 & padj<0.05) — isto é, somente padj<0.05.
    Salva intermediários por contraste.
    """
    p = f"{DIFFEXP_DIR}/{contrast}.deseq2.results.tsv"
    df = pd.read_csv(p, sep="\t")
    for col in ['gene_id','log2FoldChange','padj']:
        if col not in df.columns:
            raise KeyError(f"Faltando coluna '{col}' em {p}. Colunas: {list(df.columns)}")

    df['gene_id_base']   = df['gene_id'].astype(str).str.split('.').str[0]
    df['symbol']         = df['gene_id_base'].map(ENSG2SYM)
    df['biotype']        = df['gene_id_base'].map(ENSG2TYPE)
    df['log2FoldChange'] = pd.to_numeric(df['log2FoldChange'], errors='coerce')
    df['padj']           = pd.to_numeric(df['padj'], errors='coerce')
    df['absLFC']         = df['log2FoldChange'].abs()
    df['pass']           = df['padj'].notna() & (df['padj'] < D_FILTER_PADJ_FILTER) & (df['absLFC'] >= D_FILTER_ABSLFC_FILTER)

    # intermediários
    df.to_csv(f"{FILTER_DEG_EXPORT_DIR}/d_DEG_raw_annot_{contrast}.tsv", sep="\t", index=False)
    df.loc[df['pass'], ['gene_id','gene_id_base','symbol','log2FoldChange','padj']].to_csv(
        f"{FILTER_DEG_EXPORT_DIR}/d_DEG_filtered_{contrast}.tsv", sep="\t", index=False
    )
    return df

# -- União dos DEGs que passam nos contrastes (corrige 'All arrays…') --
deg_partes = []
for c in CONTRASTS_D:
    dfc = _carrega_deg_anotado(c)
    deg_partes.append(dfc.loc[dfc['pass'], ['gene_id_base','symbol']])

deg_union = (pd.concat(deg_partes, ignore_index=True)
             .drop_duplicates()
             .assign(symbol_up=lambda d: d['symbol'].astype(str).str.upper()))

# completa símbolos ausentes com ENSG2SYM
mask_na = deg_union['symbol'].isna() | (deg_union['symbol'].astype(str)=='') | (deg_union['symbol'].astype(str).str.lower()=='nan')
deg_union.loc[mask_na, 'symbol'] = deg_union.loc[mask_na, 'gene_id_base'].map(ENSG2SYM)
deg_union['symbol_up'] = deg_union['symbol'].astype(str).str.upper()

# exporta união (rastreamento)
deg_union.to_csv(f"{FILTER_DEG_EXPORT_DIR}/d_DEG_filtered_union.tsv", sep="\t", index=False)
(deg_union['symbol_up'].dropna().drop_duplicates().sort_values()
 ).to_csv(f"{FILTER_DEG_EXPORT_DIR}/d_DEG_filtered_union_symbols.txt", index=False, header=False)

# -- Mapeia DEGs para nós do grafo (símbolo e fallback ENSG-like) --
bn = base_nodes.copy()
bn['gene_name_up'] = bn['gene_name'].astype(str).str.upper()
bn['ensg_like']    = np.where(
    bn['gene_name_up'].str.startswith('ENSG'),
    bn['gene_name_up'].str.split('.').str[0],
    np.nan
)

deg_syms = set(deg_union['symbol_up'])
deg_ids  = set(deg_union['gene_id_base'])

nodes_por_simbolo = set(bn.loc[bn['gene_name_up'].isin(deg_syms),'nodeName'])
nodes_por_ensg    = set(bn.loc[bn['ensg_like'].isin(deg_ids),'nodeName'])
deg_nodes_in_graph = nodes_por_simbolo | nodes_por_ensg

# -- Subgrafo ESTRITO: somente arestas DEG↔DEG no grafo-base completo --
edges_d = base_edges[
    base_edges['fromNode'].isin(deg_nodes_in_graph) &
    base_edges['toNode'].isin(deg_nodes_in_graph)
].copy()

# -- Exporta + métricas (rótulos ajustados ao novo critério) --
prefix = "d_padj_lt0p05_STRICT"
rotulo = "d_padj_lt0p05_STRICT"
export_cytoscape(edges_d, base_nodes, f"{FILTER_EXPORT_DIR}/{prefix}", extra_nodes=ALWAYS_NODES)
met_d = save_metrics(rotulo, edges_d, base_nodes, extra_nodes=ALWAYS_NODES)

print(f"[D estrito] N={int(met_d.loc[0,'N'])} | E={int(met_d.loc[0,'E'])} → {FILTER_EXPORT_DIR}/{prefix}_edges.tsv/_nodes.tsv")

In [ ]:
# ===============================
# (e) ChIP-seq: RPKM (QC) + ROBUSTO=≥10 reads em ≥1 amostra → aplicar à rede + salvar intermediários
# ===============================
CHIP_COUNTS = f"{SOURCE_FOLDER}/data/processed/chip_seq_salmon.merged.transcript_counts.tsv"
chip = pd.read_csv(CHIP_COUNTS, sep='\t')

# detectar amostras e forçar numérico
ID_COLS = {'gene_id','gene_name','transcript_id','transcript_name','tx_id','tx_name','gene_id_base'}
candidates = [c for c in chip.columns if c not in ID_COLS]
num_df = chip[candidates].apply(pd.to_numeric, errors='coerce')
sample_cols_chip = [c for c in candidates if num_df[c].notna().any()]
chip[sample_cols_chip] = num_df[sample_cols_chip].fillna(0.0).astype('float64')

# --- GTF cru disponível? Se não, carregar ---
if '_gtf_raw' not in globals():
    gtf_cols = ['seqname','source','feature','start','end','score','strand','frame','attribute']
    _gtf_raw = pd.read_csv(GTF_FILE, sep='\t', comment='#', header=None, names=gtf_cols)

# --- construir tx_attr de forma SEGURA a partir do GTF cru (_gtf_raw) ---
tx_rows = _gtf_raw[_gtf_raw['feature'] == 'transcript'][['start','end','attribute']].copy()
tx_parsed = tx_rows['attribute'].apply(lambda s: parse_gtf_attributes(s, ['transcript_id','gene_id']))
tx_parsed = pd.json_normalize(tx_parsed)

tx_attr = pd.concat([tx_rows[['start','end']].reset_index(drop=True),
                     tx_parsed.reset_index(drop=True)], axis=1)

# checagens mínimas
if 'transcript_id' not in tx_attr.columns or 'gene_id' not in tx_attr.columns:
    raise ValueError("GTF não contém 'transcript_id'/'gene_id' nas linhas de 'transcript'. Verifique o arquivo GTF.")

# comprimento do transcrito e gene_id_base
tx_attr = tx_attr.dropna(subset=['transcript_id','gene_id']).copy()
tx_attr['length_bp']    = (tx_attr['end'] - tx_attr['start'] + 1).astype(float)
tx_attr['gene_id_base'] = tx_attr['gene_id'].astype(str).str.split('.').str[0]

# mapeamentos e comprimento médio por gene
tx2gene     = tx_attr.set_index('transcript_id')['gene_id_base'].to_dict()
gene_len_bp = tx_attr.groupby('gene_id_base')['length_bp'].mean()

# --- montar chip['gene_id_base'] e agregar por gene ---
if 'transcript_id' in chip.columns:
    chip['gene_id_base'] = chip['transcript_id'].map(tx2gene).fillna(
        chip.get('gene_id','').astype(str).str.split('.').str[0] if 'gene_id' in chip.columns else ''
    )
else:
    chip['gene_id_base'] = chip['gene_id'].astype(str).str.split('.').str[0]

gene_counts = chip.groupby('gene_id_base')[sample_cols_chip].sum()

# robusto = ≥10 reads em ≥1 amostra (raw)
robust_mask = (gene_counts >= 10).any(axis=1)
chip_robust_ids  = set(gene_counts.index[robust_mask])

# RPKM (QC)
gene_len_kb = (gene_len_bp/1e3).reindex(gene_counts.index)
gene_len_kb = gene_len_kb.fillna(gene_len_kb.median())
lib_sizes   = gene_counts.sum(axis=0)
rpkm = gene_counts.div(gene_len_kb, axis=0).div(lib_sizes/1e6, axis=1)


chip_robust_syms = {ENSG2SYM.get(g, g).upper() for g in chip_robust_ids}

# aplicar à rede (Cytoscape)
keep_nodes_e = set(
    base_nodes['nodeName'][
        base_nodes['gene_name'].fillna('').astype(str).str.upper().isin(chip_robust_syms)
    ]
)
edges_e = base_edges[
    base_edges['fromNode'].isin(keep_nodes_e) &
    base_edges['toNode'].isin(keep_nodes_e)
].copy()

export_cytoscape(edges_e, base_nodes, f"{FILTER_EXPORT_DIR}/e_chipseq_robust", extra_nodes=ALWAYS_NODES)
met_e = save_metrics('e_chipseq_robust', edges_e, base_nodes, extra_nodes=ALWAYS_NODES)


# intermediários (Drive)
rpkm.to_csv(f"{SUMMARY_EXPORT_DIR}/e_chipseq_gene_rpkm.tsv", sep='\t')
gene_counts.to_csv(f"{SUMMARY_EXPORT_DIR}/e_chipseq_gene_raw_counts.tsv", sep='\t')
pd.DataFrame({'gene_ENSG_base': sorted(chip_robust_ids),
              'symbol': [ENSG2SYM.get(g, g) for g in sorted(chip_robust_ids)]}
            ).to_csv(f"{SUMMARY_EXPORT_DIR}/e_chip_robust_gene_list.csv", index=False)

print(f"[ChIP] genes robustos (≥10 reads em ≥1 amostra): {len(chip_robust_syms)} | subgrafo (e): N={int(met_e.loc[0,'N'])}, E={int(met_e.loc[0,'E'])}")

In [ ]:
# ======================================================================
# (F) RECORTE CONHECIDO (GEO GSE47535) + lncRNAs 1-hop
#   - Picos GATA6 (GSM1151694) → genes (TSS ±10 kb)
#   - Atualiza literature_gata6_network_genes.txt
#   - Sub-rede: conhecidos↔conhecidos  ∪  conhecidos↔lncRNA
#   - Exporta edges/nodes + métricas
#     • GSE47535  — “Genome-wide map of GATA6 DNA binding in human PDAC cells” (ex.: GSM1151694).
#       Série: https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE47535  (peaks em supl.).
# ======================================================================

# 1) Ler peaks (precisamos de chr, start, end)
pk = pd.read_csv(PEAKS_GSE47535, sep=r"\s+", engine="python", comment="#")
pk = pk.rename(columns={pk.columns[0]:"chr", pk.columns[1]:"start", pk.columns[2]:"end"})
pk['start'] = pd.to_numeric(pk['start'], errors='coerce')
pk['end']   = pd.to_numeric(pk['end'],   errors='coerce')
pk = pk.dropna(subset=['chr','start','end']).copy()
pk['center'] = ((pk['start'] + pk['end'])/2).astype(int)

# 2) Preparar TSS por gene (do GTF cru)
gtf_cols = ['seqname','source','feature','start','end','score','strand','frame','attribute']
_gtf_raw = pd.read_csv(GTF_FILE, sep='\t', comment='#', header=None, names=gtf_cols)

g_attr = _gtf_raw[_gtf_raw['feature']=='gene'].copy()
gad = g_attr['attribute'].apply(lambda s: parse_gtf_attributes(s, ['gene_id','gene_name']))
gad = pd.json_normalize(gad)
genes = pd.concat([g_attr[['seqname','start','end','strand']].reset_index(drop=True),
                   gad.reset_index(drop=True)], axis=1)
genes['SYM'] = genes['gene_name'].astype(str).str.upper()
genes['tss'] = np.where(genes['strand']=='+', genes['start'], genes['end'])
genes_tss = genes[['seqname','tss','SYM']].dropna().rename(columns={'seqname':'chr'})

# 3) Mapear pico → gene mais próximo (TSS ±10 kb)
MAX_DIST = 10_000
mapped_syms = set()
for chrom, subp in pk.groupby('chr'):
    gsub = genes_tss[genes_tss['chr']==chrom].sort_values('tss')
    if gsub.empty:
        continue
    tss = gsub['tss'].values
    sy  = gsub['SYM'].values
    idx = np.searchsorted(tss, subp['center'].values)
    idx0 = np.clip(idx-1, 0, len(tss)-1)
    idx1 = np.clip(idx,   0, len(tss)-1)
    d0 = np.abs(subp['center'].values - tss[idx0])
    d1 = np.abs(subp['center'].values - tss[idx1])
    take_left = d0 <= d1
    nearest = np.where(take_left, sy[idx0], sy[idx1])
    dist    = np.where(take_left, d0, d1)
    mapped_syms.update([s for s,d in zip(nearest, dist) if d <= MAX_DIST])

genes_from_gse47535 = sorted(mapped_syms)
pd.Series(genes_from_gse47535).to_csv(OUT_GSE_LIST, index=False, header=False)
print(f"[GSE47535] Genes mapeados (TSS ≤10 kb): {len(genes_from_gse47535)}  -> {OUT_GSE_LIST}")

# 4) Atualizar lista principal de literatura (merge com o que já existir)
if os.path.exists(OUT_LIT_LIST):
    old = set(pd.read_csv(OUT_LIT_LIST, header=None)[0].astype(str).str.upper().str.strip())
else:
    old = set()
merged_syms = sorted(old | set(genes_from_gse47535))
pd.Series(merged_syms).to_csv(OUT_LIT_LIST, index=False, header=False)
print(f"[Literature] Lista principal atualizada: {OUT_LIT_LIST} (n={len(merged_syms)})")

# 5) Construir recorte (f): conhecidos + lncRNAs 1-hop
#    (se quiser, você pode ainda acrescentar MSigDB/TRRUST antes desta etapa)
known_syms = set(merged_syms)

# símbolo → nodeName na rede
name2node = (base_nodes.assign(SYM=base_nodes['gene_name'].fillna('').astype(str).str.upper())
                        .groupby('SYM')['nodeName'].apply(set).to_dict())
known_nodes = set().union(*[name2node.get(sym, set()) for sym in known_syms])

# sub-rede conhecido↔conhecido
edges_known2known = base_edges[
    base_edges['fromNode'].isin(known_nodes) & base_edges['toNode'].isin(known_nodes)
].copy()

# conhecido↔lncRNA (1-hop)
biotype_map = dict(zip(base_nodes['nodeName'], base_nodes['biotype']))
seed_incident = base_edges[
    base_edges['fromNode'].isin(known_nodes) | base_edges['toNode'].isin(known_nodes)
].copy()
is_seed  = seed_incident['fromNode'].isin(known_nodes)
other_n  = np.where(is_seed, seed_incident['toNode'], seed_incident['fromNode'])
lnc_mask = [biotype_map.get(n, '')=='lncRNA' for n in other_n]

edges_seed_lnc = seed_incident.loc[
    (is_seed & pd.Series(lnc_mask)) | ((~is_seed) & pd.Series(lnc_mask))
, ['fromNode','toNode','weight']].copy()

# união final do (f)
edges_f = pd.concat([edges_known2known, edges_seed_lnc], ignore_index=True).drop_duplicates()

# exportar + métricas
export_cytoscape(edges_f, base_nodes, f"{FILTER_EXPORT_DIR}/f_known_GATA6_plus_lnc_neighbors", extra_nodes=ALWAYS_NODES)
met_f = save_metrics('f_known_GATA6_plus_lnc_neighbors', edges_f, base_nodes, extra_nodes=ALWAYS_NODES)

print(f"[f/GSE47535] sementes={len(known_syms)} | presentes_na_rede={len(known_nodes)} | "
      f"subgrafo: N={int(met_f.loc[0,'N'])}, E={int(met_f.loc[0,'E'])} | saída: {FILTER_EXPORT_DIR}")

In [ ]:
# ============================================================
# (g) COMBO ESTRITO: interseção d ∩ e ∩ f (não pode exceder restritores)
#     Usa edges_e (ChIP robusto) e edges_f (literatura GATA6 + lnc 1-hop)
# ============================================================
nodes_d = set(edges_d['fromNode']) | set(edges_d['toNode'])
nodes_e = set(edges_e['fromNode']) | set(edges_e['toNode'])   # do passo (e)
nodes_f = set(edges_f['fromNode']) | set(edges_f['toNode'])   # do passo (f)

nodes_g = nodes_d & nodes_e & nodes_f
edges_g = base_edges[
    base_edges['fromNode'].isin(nodes_g) &
    base_edges['toNode'].isin(nodes_g)
].copy()

export_cytoscape(edges_g, base_nodes, f"{FILTER_EXPORT_DIR}/g_biological_combo_STRICT", extra_nodes=ALWAYS_NODES)
met_g = save_metrics('g_biological_combo_STRICT', edges_g, base_nodes, extra_nodes=ALWAYS_NODES)

print(f"[g] Combo estrito: N={int(met_g.loc[0,'N'])}, E={int(met_g.loc[0,'E'])}")

#### Enriquecimento

In [ ]:
# --------------------------------------------------------------
# ENRIQUECIMENTO FUNCIONAL (GO BP) PARA A REDE g (opcional)
#  - Por padrão NÃO usa web; sempre salva gene list p/ IPA/clusterProfiler
#  - Se quiser web, mude USE_GSEAPY=True (requer >=10 genes)
# --------------------------------------------------------------
USE_GSEAPY = True  # deixe False para não fazer chamada web agora
QTY_TERMS = 50

# nós presentes na rede g
nodes_g = base_nodes[base_nodes['nodeName'].isin(set(edges_g['fromNode']) | set(edges_g['toNode']))].copy()
# somente protein_coding para GO
symbols_g = (nodes_g.loc[nodes_g['biotype']=='protein_coding','gene_name']
                     .dropna().astype(str).str.upper().unique().tolist())

# salve SEMPRE a lista de genes para IPA/clusterProfiler
ipa_list_path = f"{FILTER_ENRICH_EXPORT_DIR}/g_gene_list_for_IPA.csv"
pd.DataFrame({'gene':symbols_g}).to_csv(ipa_list_path, index=False)

if USE_GSEAPY and len(symbols_g) >= 10:
    try:
        import gseapy as gp
        enr = gp.enrichr(gene_list=symbols_g,
                         gene_sets=['GO_Biological_Process_2023'],
                         organism='Human', outdir=None, cutoff=0.05)
        enr_res = enr.results.sort_values(['Adjusted P-value','Odds Ratio'])

        out_csv = f"{FILTER_ENRICH_EXPORT_DIR}/g_enrichr_GO_BP_top_{QTY_TERMS}.csv"
        enr_res.head(QTY_TERMS).to_csv(out_csv, index=False)

        print(f"[Enrichr] {min(QTY_TERMS, enr_res.shape[0])} termos salvos em {out_csv}")
    except Exception as e:
        print(f"[WARN] Enriquecimento não executado: {e}. Lista para IPA salva em {ipa_list_path}")
else:
    if len(symbols_g) < 10:
        print(f"[Enrich] Rede g tem poucos genes PC (n={len(symbols_g)}). Pulei Enrichr; "
              f"lista salva em {ipa_list_path}.")
    else:
        print(f"[Enrich] Web desativado. Lista para IPA salva em {ipa_list_path}.")

In [ ]:
# --------------------------------------------------------------
# TOP-5 lncRNAs e TOP-5 'genes possivelmente sem função' nos módulos associados a GATA6 (rede g)
# --------------------------------------------------------------
from collections import defaultdict

def degree_in_edges(edges_df):
    d = defaultdict(int)
    for u,v in edges_df[['fromNode','toNode']].itertuples(index=False):
        d[u]+=1; d[v]+=1
    return d

if edges_g.empty:
    print("[Top5] Rede g vazia — não há como ranquear hubs locais.")
    top5_lnc = pd.DataFrame(columns=['gene_name','module','degree'])
    top5_unknown_genes = pd.DataFrame(columns=['gene_name','module','degree'])
else:
    deg_g = degree_in_edges(edges_g)
    nodes_g = nodes_g.copy()
    nodes_g['degree'] = nodes_g['nodeName'].map(deg_g).fillna(0)

    # módulos que contêm GATA6 / GATA6-AS1 (se não houver, usa o maior módulo de g)
    mods_gata = nodes_g.loc[nodes_g['gene_name'].str.upper().isin(['GATA6','GATA6-AS1']),
                            'module'].unique().tolist()
    if not mods_gata:
        mods_gata = nodes_g['module'].value_counts().index[:1].tolist()

    sel = nodes_g[nodes_g['module'].isin(mods_gata)].copy()

    top5_lnc = (sel[sel['biotype']=='lncRNA']
                .sort_values('degree', ascending=False)
                [['gene_name','module','degree']].head(5))

    # heurística de "sem função"
    pat = re.compile(r'^(C\d+orf\d+|LOC\d+|LINC\d+|AC\d+|AL\d+)', re.I)
    unknown_mask = sel['biotype'].eq('protein_coding') & sel['gene_name'].astype(str).str.match(pat)
    top5_unknown_genes = (sel[unknown_mask]
                          .sort_values('degree', ascending=False)
                          [['gene_name','module','degree']].head(5))

# salvar
top5_lnc.to_csv(f"{FILTER_ENRICH_EXPORT_DIR}/g_top5_lncRNA_in_GATA6_modules.csv", index=False)
top5_unknown_genes.to_csv(f"{FILTER_ENRICH_EXPORT_DIR}/g_top5_unknown_genes_in_GATA6_modules.csv", index=False)

#### Relatório Rápido pra validação

In [ ]:
# ===============================
# Resumo final
# ===============================
summary = pd.concat([met_a, met_b, met_b_1, met_c, met_c_1, met_d, met_e, met_f, met_g], ignore_index=True)

summary.to_csv(f"{SUMMARY_EXPORT_DIR}/filters_summary_metrics.csv", index=False)
summary[['filter','N','E','avg_degree','density','pct_lnc','has_GATA6','has_GATA6_AS1']]

In [ ]:
# ============================================================
# CONTAGENS lncRNA vs protein_coding
# 1) Grafo ORIGINAL (CytoscapeInput-edges.txt)
# 2) ChIP-seq filtrado (E) — usa edges/nodes exportados do passo (E)
#    * inclui ALWAYS_NODES (GATA6 / GATA6-AS1) nos conjuntos de nós
# ============================================================

# --- helpers rápidos ---
def _counts_global(nodes_df):
    cg = (nodes_df['biotype'].value_counts()
          .reindex(['lncRNA','protein_coding'], fill_value=0)
          .rename_axis('biotype').reset_index(name='n'))
    return cg

def _counts_by_module(nodes_df):
    cm = (nodes_df.groupby(['module','biotype']).size()
          .unstack(fill_value=0)
          .reindex(columns=['lncRNA','protein_coding'], fill_value=0)
          .reset_index())
    return cm

# ------------------------------------------------------------
# 1) GRAFO ORIGINAL
# ------------------------------------------------------------
orig_node_set = set(base_edges['fromNode']) | set(base_edges['toNode'])
# força presença dos ALWAYS (sem duplicar)
if 'ALWAYS_NODES' in globals() and isinstance(ALWAYS_NODES, set):
    orig_node_set |= ALWAYS_NODES

orig_nodes = base_nodes[base_nodes['nodeName'].isin(orig_node_set)].copy()

orig_counts_global = _counts_global(orig_nodes)
orig_counts_by_mod = _counts_by_module(orig_nodes)

# salvar
orig_counts_global.to_csv(f"{SUMMARY_EXPORT_DIR}/original_counts_biotype_global.csv", index=False)
orig_counts_by_mod.to_csv(f"{SUMMARY_EXPORT_DIR}/original_counts_biotype_by_module.csv", index=False)

# imprimir resumo
n_lnc = int(orig_counts_global.loc[orig_counts_global['biotype']=='lncRNA','n'])
n_pc  = int(orig_counts_global.loc[orig_counts_global['biotype']=='protein_coding','n'])
print(f"[ORIGINAL] nós={orig_nodes.shape[0]} | lncRNA={n_lnc} | protein_coding={n_pc}")
print("[ORIGINAL] Top módulos por nº de nós:")
display(orig_counts_by_mod.sort_values(['lncRNA','protein_coding'], ascending=False).head(10))

# ------------------------------------------------------------
# 2) ChIP-seq FILTRADO (E) — garante arquivo e contagens
#    Arquivos esperados do passo (E):
#     - {FILTER_EXPORT_DIR}/e_chipseq_robust_edges.tsv
#     - {FILTER_EXPORT_DIR}/e_chipseq_robust_nodes.tsv
# ------------------------------------------------------------
E_PREFIX = f"{FILTER_EXPORT_DIR}/e_chipseq_robust"
if 'edges_e' not in globals() or 'base_nodes' not in globals():
    # re-carrega se necessário
    edges_e = pd.read_csv(f"{E_PREFIX}_edges.tsv", sep="\t")
    # nodes base já está em base_nodes; os nodes do recorte serão filtrados abaixo

nodes_e_set = set(edges_e['fromNode']) | set(edges_e['toNode'])
# força presença dos ALWAYS (sem duplicar)
if 'ALWAYS_NODES' in globals() and isinstance(ALWAYS_NODES, set):
    nodes_e_set |= ALWAYS_NODES

nodes_e_df = base_nodes[base_nodes['nodeName'].isin(nodes_e_set)].copy()

chip_counts_global = _counts_global(nodes_e_df)
chip_counts_by_mod = _counts_by_module(nodes_e_df)

# salvar contagens
chip_counts_global.to_csv(f"{SUMMARY_EXPORT_DIR}/e_counts_biotype_global.csv", index=False)
chip_counts_by_mod.to_csv(f"{SUMMARY_EXPORT_DIR}/e_counts_biotype_by_module.csv", index=False)

# imprimir resumo + caminhos úteis
e_lnc = int(chip_counts_global.loc[chip_counts_global['biotype']=='lncRNA','n'])
e_pc  = int(chip_counts_global.loc[chip_counts_global['biotype']=='protein_coding','n'])
print(f"[ChIP E] nós={nodes_e_df.shape[0]} | lncRNA={e_lnc} | protein_coding={e_pc}")
print("[ChIP E] Top módulos por nº de nós:")
display(chip_counts_by_mod.sort_values(['lncRNA','protein_coding'], ascending=False).head(10))

print("\n[Arquivos do ChIP-seq filtrado (E) já gerados/esperados]:")
print(" - edges:", f"{E_PREFIX}_edges.tsv")
print(" - nodes:", f"{E_PREFIX}_nodes.tsv")
print(" - contagens globais:", f"{FILTER_EXPORT_DIR}/e_counts_biotype_global.csv")
print(" - contagens por módulo:", f"{FILTER_EXPORT_DIR}/e_counts_biotype_by_module.csv")

#### IntaRNA

In [ ]:
# ============================================================
# IntaRNA: geração de insumos (FASTA) e catálogo de pares
#  - Queries  : lncRNAs (top-k por grau na rede G)
#  - Targets  : mRNAs (vizinhos protein_coding por peso, top-N)
#  - Sequência: transcrito "spliced" (concatenação de exons) do GTF + genoma
#  - Saídas   : FASTA combinados, FASTA por lncRNA, catálogo de pares, script-exemplo
# ============================================================

# Configurações para o INTA
TOP_K_LNCS       = 5    # nº de lncRNAs (queries)
TOP_N_TARGETS    = 10   # nº de mRNAs por lnc (targets)
ALWAYS_INCLUDE   = ['GATA6-AS1']  # força inclusão se existir no grafo G
MIN_EDGE_WEIGHT  = None           # opcional: filtrar alvos por peso mínimo (None = desativado)


In [ ]:
# -----------------------------
# Seleção de lncRNAs e mRNAs alvo a partir de G
# -----------------------------
if 'edges_g' not in globals() or edges_g.empty:
    raise RuntimeError("edges_g não está definido/vazio. Execute antes a etapa (g) que constrói o combo estrito G.")

# Subconjunto de nós presentes em G
nodes_in_G = set(edges_g['fromNode']).union(set(edges_g['toNode']))
nodesG = base_nodes[base_nodes['nodeName'].isin(nodes_in_G)].copy()

# Grau na rede G
def _degree_from_edges(edf: pd.DataFrame):
    d = defaultdict(int)
    for u,v in edf[['fromNode','toNode']].itertuples(index=False):
        d[u]+=1; d[v]+=1
    return d
degG = _degree_from_edges(edges_g)
nodesG['degree_G'] = nodesG['nodeName'].map(degG).fillna(0).astype(int)

# mapeamentos auxiliares
biotype_map  = dict(zip(base_nodes['nodeName'], base_nodes['biotype']))
name2sym     = dict(zip(base_nodes['nodeName'], base_nodes['gene_name'].astype(str)))
sym2node_set = (base_nodes
                .assign(SYM=base_nodes['gene_name'].astype(str).str.upper())
                .groupby('SYM')['nodeName'].apply(set).to_dict())

# Seleciona lncRNAs candidatos (top-K por grau) e garante ALWAYS_INCLUDE se existir no grafo
lncs_G = nodesG[nodesG['biotype'].eq('lncRNA')].copy()
lncs_G['sym_up'] = lncs_G['gene_name'].astype(str).str.upper()
top_lncs = (lncs_G.sort_values('degree_G', ascending=False)
                  .drop_duplicates('sym_up')
                  .head(TOP_K_LNCS)['sym_up'].tolist())

for sym in ALWAYS_INCLUDE:
    SU = sym.upper()
    if SU in lncs_G['sym_up'].values and SU not in top_lncs:
        top_lncs.append(SU)

# Para cada lncRNA, escolhe top-N mRNAs vizinhos por peso
edges_g_loc = edges_g.copy()
targets_per_lnc = {}
for lnc_sym in top_lncs:
    # todos os nós (nodeName) que representam esse símbolo
    candidate_nodes = sym2node_set.get(lnc_sym, set()) & nodes_in_G
    if not candidate_nodes:
        continue

    # pega vizinhos e pesos
    # coletar arestas incidentes
    inc = edges_g_loc[(edges_g_loc['fromNode'].isin(candidate_nodes)) | (edges_g_loc['toNode'].isin(candidate_nodes))].copy()
    if MIN_EDGE_WEIGHT is not None:
        inc = inc[inc['weight'] >= MIN_EDGE_WEIGHT]

    # normaliza direção: "lnc_node" -> "other"
    is_from = inc['fromNode'].isin(candidate_nodes)
    other  = np.where(is_from, inc['toNode'], inc['fromNode'])
    inc['other'] = other

    # filtra apenas targets protein_coding
    inc['other_biotype'] = inc['other'].map(biotype_map)
    inc_pc = inc[inc['other_biotype'].eq('protein_coding')].copy()
    if inc_pc.empty:
        targets_per_lnc[lnc_sym] = []
        continue

    # agrega por gene alvo (por símbolo) usando peso máximo
    inc_pc['other_sym'] = inc_pc['other'].map(name2sym).astype(str)
    agg = (inc_pc.groupby('other_sym', as_index=False)['weight'].max()
                  .sort_values('weight', ascending=False)
                  .head(TOP_N_TARGETS))
    targets_per_lnc[lnc_sym] = agg

In [ ]:
# ------------------------------------------------------------------
# DIAGNÓSTICO / RESUMO DO QUE SERÁ ENVIADO AO INTARNA
# (insira após montar 'targets_per_lnc')
# ------------------------------------------------------------------

ONLY_CONNECTED = True  # reforça a lógica atual: só pares com aresta na rede G

# métricas agregadas
n_lncs_total = len(top_lncs)
n_lncs_with_targets = 0
n_pairs = 0
targets_unique = set()

summary_rows = []
for lnc_sym in top_lncs:
    df = targets_per_lnc.get(lnc_sym)
    if isinstance(df, pd.DataFrame) and not df.empty:
        n_t = int(df.shape[0])
        n_lncs_with_targets += 1
        n_pairs += n_t
        syms = df['other_sym'].astype(str).tolist()
        targets_unique.update(syms)
        # top-5 por peso para inspeção
        top5 = (df.sort_values('weight', ascending=False)
                  .head(5)[['other_sym','weight']].values.tolist())
    else:
        n_t = 0
        syms = []
        top5 = []
    summary_rows.append({
        'lncRNA': lnc_sym,
        'n_targets': n_t,
        'top5_targets_by_weight': ';'.join([f"{s}:{w:.3g}" for s,w in top5])
    })

summary_df = pd.DataFrame(summary_rows).sort_values('n_targets', ascending=False)
summary_csv = f"{SUMMARY_EXPORT_DIR}/intaRNA_pairs_summary.csv"
summary_df.to_csv(summary_csv, index=False)

n_targets_unique = len(targets_unique)

print("[IntaRNA] Somente pares com conexão na rede G? ->", ONLY_CONNECTED)
print(f"[IntaRNA] lncRNAs (queries) selecionados: {n_lncs_total}")
print(f"[IntaRNA] lncRNAs com ≥1 mRNA alvo:       {n_lncs_with_targets}")
print(f"[IntaRNA] Pares lnc–mRNA (total):          {n_pairs}")
print(f"[IntaRNA] mRNAs únicos (targets):          {n_targets_unique}")
print(f"[IntaRNA] Resumo salvo em: {summary_csv}")

# opcional: mostrar os 10 lncRNA mais "pesados" (mais alvos), útil para entender tempo de execução
display(summary_df.head(10))

In [ ]:
# -----------------------------
# Transcritos e extração de sequência spliced (GTF + genoma)
# -----------------------------
# Lê GTF e extrai exons por transcript_id
gtf_cols = ['seqname','source','feature','start','end','score','strand','frame','attribute']
_gtf = pd.read_csv(GTF_FILE, sep='\t', comment='#', header=None, names=gtf_cols)

attr = _gtf['attribute'].apply(lambda s: parse_gtf_attributes(s, ['gene_id','gene_name','gene_type','transcript_id']))
attr = pd.json_normalize(attr)
g = pd.concat([_gtf.drop(columns=['attribute']), attr], axis=1)

exons = g[g['feature'].eq('exon')].dropna(subset=['transcript_id'])
exons['gene_name_up'] = exons['gene_name'].astype(str).str.upper()
exons['start'] = exons['start'].astype(int)
exons['end']   = exons['end'].astype(int)

# comprimento do transcrito
tx_len = (exons.assign(len=lambda d: d['end']-d['start']+1)
               .groupby('transcript_id', as_index=False)['len'].sum())

tx2gene  = exons.groupby('transcript_id', as_index=False).first()[['transcript_id','gene_name_up','gene_type','seqname','strand']]
# transcritos por gene (para escolher o maior)
tx_by_gene = tx2gene.merge(tx_len, on='transcript_id', how='left')

# melhor transcrito = maior soma de éxons por gene
best_tx = (tx_by_gene.sort_values(['gene_name_up','len'], ascending=[True,False])
                     .drop_duplicates('gene_name_up')
                     .set_index('gene_name_up')['transcript_id']
                     .to_dict())

# indexa exons por transcrito para extração
exons_by_tx = (exons
               .sort_values(['transcript_id','start','end'])
               .groupby('transcript_id')
               .apply(lambda d: d[['seqname','strand','start','end']].values.tolist())
               .to_dict())

# carrega genoma
with gzip.open(GENOME_FA, 'rt') as handle:
    genome = SeqIO.to_dict(SeqIO.parse(handle, 'fasta'))
available_chroms = set(genome.keys())

def fetch_spliced_txseq(tx_id: str) -> str | None:
    """Concatena éxons (1-based, inclusive) e retorna sequência RNA (T→U)."""
    exs = exons_by_tx.get(tx_id)
    if not exs:
        return None
    pieces = []
    chrom = None; strand = '+'
    for (seqname, strand, s, e) in exs:
        chrom = seqname
        if chrom not in available_chroms and chrom.startswith('chr') and chrom[3:] in available_chroms:
            chrom = chrom[3:]
        if chrom not in available_chroms and f'chr{chrom}' in available_chroms:
            chrom = f'chr{chrom}'
        if chrom not in available_chroms:
            return None
        # Biopython: [s-1:e] (end exclusivo), GTF é 1-based inclusive
        pieces.append(str(genome[chrom].seq[s-1:e]).upper())
    seq = ''.join(pieces)
    if strand == '-':
        seq = str(Seq(seq).reverse_complement())
    return seq.replace('T','U')

In [ ]:
# -----------------------------
# Montagem dos FASTAs e catálogo
# -----------------------------
# universo de símbolos alvo (mRNAs) e lncRNAs
all_target_syms = set()
for lnc_sym, df in targets_per_lnc.items():
    if isinstance(df, pd.DataFrame) and not df.empty:
        all_target_syms |= set(df['other_sym'].astype(str))

# escolhe um melhor transcrito por símbolo (lnc e mRNA)
selected_lncs = []
for sym in top_lncs:
    tx = best_tx.get(sym)
    if tx: selected_lncs.append((sym, tx))

selected_targets = []
for sym in sorted(all_target_syms):
    tx = best_tx.get(sym)
    if tx: selected_targets.append((sym, tx))

# gera FASTA combinados
def write_fasta(recs, path):
    with open(path, 'w') as fh:
        for sym, tx in recs:
            seq = fetch_spliced_txseq(tx)
            if not seq:
                continue
            fh.write(f">{sym}|{tx}\n")
            # quebra em linhas de 80 col
            for i in range(0, len(seq), 80):
                fh.write(seq[i:i+80] + "\n")

combined_query_fa  = f"{INTA_EXPORT_DIR}/queries_lncRNA_top{TOP_K_LNCS}.fa"
combined_target_fa = f"{INTA_EXPORT_DIR}/targets_mRNA_union_top{TOP_N_TARGETS}.fa"

write_fasta(selected_lncs, combined_query_fa)
write_fasta(selected_targets, combined_target_fa)

# FASTA por lncRNA (um query + seus alvos)
pairs_catalog_rows = []
for lnc_sym in top_lncs:
    lnc_tx = best_tx.get(lnc_sym)
    if not lnc_tx:
        continue

    # targets desse lnc
    tgt_df = targets_per_lnc.get(lnc_sym)
    if not isinstance(tgt_df, pd.DataFrame) or tgt_df.empty:
        continue

    subdir = f"{INTA_EXPORT_DIR}/{lnc_sym}"
    os.makedirs(subdir, exist_ok=True)
    q_fa = f"{subdir}/{lnc_sym}_query.fa"
    t_fa = f"{subdir}/{lnc_sym}_targets.fa"

    write_fasta([(lnc_sym, lnc_tx)], q_fa)
    # targets deste lnc
    tgt_list = []
    for sym, w in tgt_df[['other_sym','weight']].itertuples(index=False):
        tx = best_tx.get(sym)
        if not tx:
            continue
        tgt_list.append((sym, tx))
        # guarda linha do catálogo
        # módulos (se existirem)
        # pega um nodeName de cada símbolo para buscar módulo
        node_lnc = next(iter(sym2node_set.get(lnc_sym, [])), None)
        node_tgt = next(iter(sym2node_set.get(sym, [])), None)
        mod_lnc = base_nodes.loc[base_nodes['nodeName'].eq(node_lnc),'module'].iloc[0] if node_lnc else np.nan
        mod_tgt = base_nodes.loc[base_nodes['nodeName'].eq(node_tgt),'module'].iloc[0] if node_tgt else np.nan
        pairs_catalog_rows.append({
            'lncRNA': lnc_sym,
            'lnc_tx': lnc_tx,
            'target_gene': sym,
            'target_tx': tx,
            'edge_weight': w,
            'module_lnc': mod_lnc,
            'module_target': mod_tgt,
            'query_fa': q_fa,
            'targets_fa': t_fa
        })
    if tgt_list:
        write_fasta(tgt_list, t_fa)

# catálogo global
pairs_catalog = pd.DataFrame(pairs_catalog_rows)
pairs_catalog_path = f"{INTA_EXPORT_DIR}/intaRNA_pairs_catalog.csv"
pairs_catalog.to_csv(pairs_catalog_path, index=False)

# Script com exemplos de execução (um por lncRNA)
run_sh = f"{INTA_EXPORT_DIR}/run_intarna_examples.sh"
with open(run_sh, 'w') as fh:
    fh.write("#!/usr/bin/env bash\nset -euo pipefail\n\n")
    fh.write("# Exemplos de execução do IntaRNA (um CSV por lncRNA)\n")
    fh.write("# Ajuste --outCsvCols conforme necessidade; veja 'Customizable CSV RNA-RNA interaction output'\n\n")
    for lnc_sym in top_lncs:
        subdir = f"{INTA_EXPORT_DIR}/{lnc_sym}"
        q_fa = f"{subdir}/{lnc_sym}_query.fa"
        t_fa = f"{subdir}/{lnc_sym}_targets.fa"
        if not (os.path.exists(q_fa) and os.path.exists(t_fa)):
            continue
        out_csv = f"{subdir}/{lnc_sym}_intaRNA.csv"
        # colunas sugeridas: ids, energia, posições
        cols = "id1,id2,E,hybridDP,start1,end1,start2,end2,ed1,ed2,seedStart1,seedEnd1,seedStart2,seedEnd2"
        fh.write(
            f'IntaRNA -q "{q_fa}" -t "{t_fa}" '
            f'--outMode=C --outCsvCols={cols} --out="{out_csv}" '
            f'--temperature=37 --seedBP=7\n'
        )

In [ ]:
# ============================================================
# ANÁLISE QUANTITATIVA DE DEGs (para os slides)
#   - Critério: padj < 0.05 e |log2FC| >= 2.0
#   - Saídas:
#       • Tabelas resumo por contraste (contagens globais e por biotipo)
#       • Listas de DEGs por contraste (símbolo/ENSG/biotipo/direção)
#       • Gráficos (barras) por contraste: up vs down x (lncRNA vs protein_coding)
# ============================================================
PADJ_THR = 0.05
LFC_THR  = 2.0
CONTRASTS_DE = ["siGATA6_vs_siCtl_Veh", "siGATA6_vs_siCtl_DEX"]

# ---------- util: mapa ENSG_base -> gene_name / gene_type (GENCODE v46) ----------
_gtf_cols = ['seqname','source','feature','start','end','score','strand','frame','attribute']
_gtf_raw  = pd.read_csv(GTF_FILE, sep="\t", comment="#", header=None, names=_gtf_cols)

attr = _gtf_raw[_gtf_raw['feature']=='gene']['attribute'].apply(
    lambda s: parse_gtf_attributes(s, ['gene_id','gene_name','gene_type'])
)
attr = pd.json_normalize(attr).dropna(subset=['gene_id'])
attr['gene_id_base'] = attr['gene_id'].astype(str).str.split('.').str[0]
ENSG2SYM  = dict(zip(attr['gene_id_base'], attr['gene_name'].astype(str)))
ENSG2TYPE = dict(zip(attr['gene_id_base'], attr['gene_type'].astype(str)))

def _biotype_bucket(s):
    s = (s or '').lower()
    if s == 'lncrna': return 'lncRNA'
    if s == 'protein_coding': return 'protein_coding'
    return 'others'

# ---------- função para carregar e resumir um contraste ----------
def summarize_contrast(contrast: str) -> dict:
    p = f"{DIFFEXP_DIR}/{contrast}.deseq2.results.tsv"
    df = pd.read_csv(p, sep="\t")

    # colunas esperadas
    for c in ['gene_id','log2FoldChange','padj']:
        if c not in df.columns:
            raise KeyError(f"Coluna '{c}' ausente em {p}. Colunas: {list(df.columns)}")

    # anotação
    df['gene_id_base']   = df['gene_id'].astype(str).str.split('.').str[0]
    df['symbol']         = df['gene_id_base'].map(ENSG2SYM)
    df['gene_type']      = df['gene_id_base'].map(ENSG2TYPE)
    df['biotype_cat']    = df['gene_type'].apply(_biotype_bucket)
    df['log2FoldChange'] = pd.to_numeric(df['log2FoldChange'], errors='coerce')
    df['padj']           = pd.to_numeric(df['padj'], errors='coerce')

    # filtro DEG
    df['is_deg'] = df['padj'].notna() & (df['padj'] < PADJ_THR) & (df['log2FoldChange'].abs() >= LFC_THR)
    df['direction'] = np.where(df['log2FoldChange'] >= LFC_THR, 'up',
                        np.where(df['log2FoldChange'] <= -LFC_THR, 'down', 'none'))

    deg = df[df['is_deg']].copy()

    # contagens globais
    total_all   = int(deg.shape[0])
    total_up    = int((deg['direction']=='up').sum())
    total_down  = int((deg['direction']=='down').sum())

    # contagens por biotipo
    def count_biotype(bio):
        sub = deg[deg['biotype_cat']==bio]
        return int(sub.shape[0]), int((sub['direction']=='up').sum()), int((sub['direction']=='down').sum())

    lnc_all, lnc_up, lnc_down = count_biotype('lncRNA')
    pc_all,  pc_up,  pc_down  = count_biotype('protein_coding')
    oth_all, oth_up, oth_down = count_biotype('others')

    # salva lista de DEGs do contraste
    out_list = (deg[['gene_id_base','symbol','gene_type','biotype_cat','log2FoldChange','padj','direction']]
                .sort_values(['biotype_cat','direction','padj']))
    out_list_path = f"{DEG_SUMMARY_DIR}/DEG_list_{contrast}_padj{PADJ_THR}_absLFC{LFC_THR}.tsv"
    out_list.to_csv(out_list_path, sep="\t", index=False)

    # gráfico (barras) lncRNA vs protein_coding x (up/down)
    plot_df = (deg[deg['biotype_cat'].isin(['lncRNA','protein_coding'])]
                   .groupby(['biotype_cat','direction']).size().reset_index(name='n'))
    plot_df = plot_df[plot_df['direction'].isin(['up','down'])]
    plt.figure(figsize=(7,5))
    sns.barplot(data=plot_df, x='biotype_cat', y='n', hue='direction',
                palette={'up':'#f58518','down':'#4c78a8'})
    plt.title(f"{contrast}: DEGs (padj<{PADJ_THR}, |log2FC|≥{LFC_THR})")
    plt.xlabel('')
    plt.ylabel('Número de genes')
    plt.tight_layout()
    fig_path = f"{DEG_SUMMARY_DIR}/DEG_bar_{contrast}_padj{PADJ_THR}_absLFC{LFC_THR}.png"
    plt.savefig(fig_path, dpi=300)
    plt.close()

    # tabela resumo (uma linha)
    summary_row = {
        'contrast': contrast,
        'padj_thr': PADJ_THR,
        'absLFC_thr': LFC_THR,
        'total_DEG': total_all,
        'total_up': total_up,
        'total_down': total_down,
        'lncRNA_DEG': lnc_all,
        'lncRNA_up': lnc_up,
        'lncRNA_down': lnc_down,
        'protein_coding_DEG': pc_all,
        'protein_coding_up': pc_up,
        'protein_coding_down': pc_down,
        'others_DEG': oth_all,
        'others_up': oth_up,
        'others_down': oth_down,
        'deg_list_path': out_list_path,
        'deg_barplot_path': fig_path
    }
    return summary_row, df  # retorna também o dataframe bruto anotado

# ---------- executar para os contrastes e montar o resumo ----------
all_rows = []
all_deg_pass = []
for c in CONTRASTS_DE:
    row, df_anno = summarize_contrast(c)
    all_rows.append(row)
    all_deg_pass.append(df_anno[df_anno['is_deg']][['gene_id_base','symbol','biotype_cat','direction']].assign(contrast=c))

summary_df = pd.DataFrame(all_rows)
summary_path = f"{FILTER_DEG_EXPORT_DIR}/DEG_summary_by_contrast_padj{PADJ_THR}_absLFC{LFC_THR}.csv"
summary_df.to_csv(summary_path, index=False)

print("== Resumo por contraste ==")
display(summary_df[['contrast','total_DEG','lncRNA_DEG','protein_coding_DEG','total_up','total_down']])

# ---------- (opcional) união de DEGs nos dois contrastes (contagem agregada, sem direção) ----------
deg_union = (pd.concat(all_deg_pass, ignore_index=True)
             .drop_duplicates(subset=['gene_id_base']))
deg_union_counts = (deg_union.groupby('biotype_cat').size()
                    .reindex(['lncRNA','protein_coding','others'], fill_value=0)
                    .rename('n').reset_index())
deg_union_counts_path = f"{FILTER_DEG_EXPORT_DIR}/DEG_union_counts_padj{PADJ_THR}_absLFC{LFC_THR}.csv"
deg_union_counts.to_csv(deg_union_counts_path, index=False)

print("\n== União dos DEGs (sem direção) ==")
display(deg_union_counts)

# (opcional) gráfico da união
plt.figure(figsize=(6,4))
sns.barplot(data=deg_union_counts, x='biotype_cat', y='n',
            palette={'protein_coding':'#4c78a8','lncRNA':'#f58518','others':'#54a24b'})
plt.title(f"União DEGs (padj<{PADJ_THR}, |log2FC|≥{LFC_THR})")
plt.xlabel('')
plt.ylabel('Número de genes')
plt.tight_layout()
plt.savefig(f"{FILTER_DEG_EXPORT_DIR}/DEG_union_bar_padj{PADJ_THR}_absLFC{LFC_THR}.png", dpi=300)
plt.close()

print("\n[Arquivos gerados]")
print(" - Resumo por contraste:", summary_path)
for p in summary_df['deg_list_path']:
    print(" - Lista DEG:", p)
for p in summary_df['deg_barplot_path']:
    print(" - Gráfico  :", p)
print(" - União (contagens):", deg_union_counts_path)


In [ ]:
# ============================================================
# ANÁLISE QUANTITATIVA DE DEGs (padj-only, sem |log2FC|>=2)
#   - Critério: padj < 0.05
#   - Direção: 'up' se log2FC > 0 ; 'down' se log2FC < 0 ; (log2FC==0 fica como 'zero' e é ignorado nas contagens)
#   - Saídas:
#       • Tabelas resumo por contraste (globais e por biotipo)
#       • Listas de DEGs por contraste (símbolo/ENSG/biotipo/direção)
#       • Gráficos (barras) por contraste: up vs down × (lncRNA vs protein_coding)
# ============================================================

import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

PADJ_THR = 0.05
CONTRASTS_DE = ["siGATA6_vs_siCtl_Veh", "siGATA6_vs_siCtl_DEX"]

# ---------- mapa ENSG_base -> gene_name / gene_type (GENCODE v46) ----------
_gtf_cols = ['seqname','source','feature','start','end','score','strand','frame','attribute']
_gtf_raw  = pd.read_csv(GTF_FILE, sep="\t", comment="#", header=None, names=_gtf_cols)

attr = _gtf_raw[_gtf_raw['feature']=='gene']['attribute'].apply(
    lambda s: parse_gtf_attributes(s, ['gene_id','gene_name','gene_type'])
)
attr = pd.json_normalize(attr).dropna(subset=['gene_id'])
attr['gene_id_base'] = attr['gene_id'].astype(str).str.split('.').str[0]
ENSG2SYM  = dict(zip(attr['gene_id_base'], attr['gene_name'].astype(str)))
ENSG2TYPE = dict(zip(attr['gene_id_base'], attr['gene_type'].astype(str)))

def _biotype_bucket(s):
    s = (s or '').lower()
    if s == 'lncrna': return 'lncRNA'
    if s == 'protein_coding': return 'protein_coding'
    return 'others'

def summarize_contrast_padj_only(contrast: str) -> dict:
    """Carrega um contraste, aplica padj<0.05, classifica por sinal do LFC e salva saídas."""
    p = f"{DIFFEXP_DIR}/{contrast}.deseq2.results.tsv"
    df = pd.read_csv(p, sep="\t")

    # checagem de colunas
    for c in ['gene_id','log2FoldChange','padj']:
        if c not in df.columns:
            raise KeyError(f"Coluna '{c}' ausente em {p}. Colunas: {list(df.columns)}")

    # anotação
    df['gene_id_base']   = df['gene_id'].astype(str).str.split('.').str[0]
    df['symbol']         = df['gene_id_base'].map(ENSG2SYM)
    df['gene_type']      = df['gene_id_base'].map(ENSG2TYPE)
    df['biotype_cat']    = df['gene_type'].apply(_biotype_bucket)
    df['log2FoldChange'] = pd.to_numeric(df['log2FoldChange'], errors='coerce')
    df['padj']           = pd.to_numeric(df['padj'], errors='coerce')

    # filtro padj-only
    df['is_deg']   = df['padj'].notna() & (df['padj'] < PADJ_THR)
    df['direction'] = np.where(df['log2FoldChange'] > 0, 'up',
                        np.where(df['log2FoldChange'] < 0, 'down', 'zero'))

    # para contagens, ignorar 'zero'
    deg = df[df['is_deg'] & df['direction'].isin(['up','down'])].copy()

    # contagens globais
    total_all   = int(deg.shape[0])
    total_up    = int((deg['direction']=='up').sum())
    total_down  = int((deg['direction']=='down').sum())

    # contagens por biotipo
    def count_biotype(bio):
        sub = deg[deg['biotype_cat']==bio]
        return int(sub.shape[0]), int((sub['direction']=='up').sum()), int((sub['direction']=='down').sum())

    lnc_all, lnc_up, lnc_down = count_biotype('lncRNA')
    pc_all,  pc_up,  pc_down  = count_biotype('protein_coding')
    oth_all, oth_up, oth_down = count_biotype('others')

    # salva lista de DEGs do contraste
    out_list = (deg[['gene_id_base','symbol','gene_type','biotype_cat','log2FoldChange','padj','direction']]
                .sort_values(['biotype_cat','direction','padj']))
    out_list_path = f"{DEG_SUMMARIES_EXPORT_DIR}/DEG_padjOnly_list_{contrast}_padj{PADJ_THR}.tsv"
    out_list.to_csv(out_list_path, sep="\t", index=False)

    # gráfico (barras) lncRNA vs protein_coding x (up/down)
    plot_df = (deg[deg['biotype_cat'].isin(['lncRNA','protein_coding'])]
                   .groupby(['biotype_cat','direction']).size().reset_index(name='n'))
    plt.figure(figsize=(7,5))
    sns.barplot(data=plot_df, x='biotype_cat', y='n', hue='direction',
                palette={'up':'#f58518','down':'#4c78a8'})
    plt.title(f"{contrast}: DEGs (padj<{PADJ_THR}, sem |log2FC| mínimo)")
    plt.xlabel('')
    plt.ylabel('Número de genes')
    plt.tight_layout()
    fig_path = f"{DEG_SUMMARIES_EXPORT_DIR}/DEG_padjOnly_bar_{contrast}_padj{PADJ_THR}.png"
    plt.savefig(fig_path, dpi=300)
    plt.close()

    # linha de resumo
    summary_row = {
        'contrast': contrast,
        'padj_thr': PADJ_THR,
        'total_DEG_padjOnly': total_all,
        'total_up': total_up,
        'total_down': total_down,
        'lncRNA_DEG': lnc_all,
        'lncRNA_up': lnc_up,
        'lncRNA_down': lnc_down,
        'protein_coding_DEG': pc_all,
        'protein_coding_up': pc_up,
        'protein_coding_down': pc_down,
        'others_DEG': oth_all,
        'others_up': oth_up,
        'others_down': oth_down,
        'deg_list_path': out_list_path,
        'deg_barplot_path': fig_path
    }
    return summary_row, df

# ---------- executar e montar resumo ----------
all_rows = []
all_deg_pass = []
for c in CONTRASTS_DE:
    row, df_raw = summarize_contrast_padj_only(c)
    all_rows.append(row)
    # guardar união sem direção (se quiser um agregado geral)
    all_deg_pass.append(
        df_raw[df_raw['is_deg']][['gene_id_base','symbol','biotype_cat']].assign(contrast=c)
    )

summary_df = pd.DataFrame(all_rows)
summary_path = f"{DEG_SUMMARIES_EXPORT_DIR}/DEG_padjOnly_summary_by_contrast_padj{PADJ_THR}.csv"
summary_df.to_csv(summary_path, index=False)

print("== Resumo por contraste (padj-only) ==")
display(summary_df[['contrast','total_DEG_padjOnly','lncRNA_DEG','protein_coding_DEG','total_up','total_down']])

# União (padj-only, sem direção)
deg_union = (pd.concat(all_deg_pass, ignore_index=True)
             .drop_duplicates(subset=['gene_id_base']))
deg_union_counts = (deg_union.groupby('biotype_cat').size()
                    .reindex(['lncRNA','protein_coding','others'], fill_value=0)
                    .rename('n').reset_index())
deg_union_counts_path = f"{DEG_SUMMARIES_EXPORT_DIR}/DEG_padjOnly_union_counts_padj{PADJ_THR}.csv"
deg_union_counts.to_csv(deg_union_counts_path, index=False)

print("\n== União dos DEGs padj-only (sem direção) ==")
display(deg_union_counts)

# (opcional) gráfico da união
plt.figure(figsize=(6,4))
sns.barplot(data=deg_union_counts, x='biotype_cat', y='n',
            palette={'protein_coding':'#4c78a8','lncRNA':'#f58518','others':'#54a24b'})
plt.title(f"União DEGs padj-only (padj<{PADJ_THR})")
plt.xlabel('')
plt.ylabel('Número de genes')
plt.tight_layout()
plt.savefig(f"{DEG_SUMMARIES_EXPORT_DIR}/DEG_padjOnly_union_bar_padj{PADJ_THR}.png", dpi=300)
plt.close()

print("\n[Arquivos gerados — padj-only]")
print(" - Resumo por contraste:", summary_path)
for p in summary_df['deg_list_path']:
    print(" - Lista DEG:", p)
for p in summary_df['deg_barplot_path']:
    print(" - Gráfico  :", p)
print(" - União (contagens):", deg_union_counts_path)